# AI-First LEO CDP — RAG Test Suite (9 Business Use Cases)

This notebook builds one shared RAG toolkit and then tests it across eight real workloads for a Customer Data Platform (CDP). Every use case below follows the same documentation contract: **Description**, **Flow**, **Input metadata**, **Output metadata** — so you can read any one section on its own and know exactly what goes in and what comes out.

| #     | Use case                                                        | Model(s)                                                    | Description                                                                                                                                                                                                                                                                                                                                                                                                                 |
| ----- | --------------------------------------------------------------- | ----------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1     | Customer Support                                                | `deepseek/deepseek-chat`                                    | RAG over a help-center knowledge base, with a Tavily domain-restricted web search fallback when the answer cannot be found in the KB                                                                                                                                                                                                                                                                                        |
| 2     | Text-to-SQL                                                     | `deepseek/deepseek-v4-pro`                                  | Schema-grounded RAG → SQL generation → SQL execution against a small sample CDP database                                                                                                                                                                                                                                                                                                                                    |
| 3     | Marketing Automation — Campaign Orchestration                   | `moonshotai/kimi-k3`                                        | A RAG-grounded, tool-calling agent that plans and executes a multi-step campaign workflow                                                                                                                                                                                                                                                                                                                                   |
| 4     | Marketing Automation — Copy Drafting                            | `~deepseek/deepseek-v4-flash-latest`                              | Brand-guideline-grounded structured copy generation, including subject line, preview text, email body, and social caption                                                                                                                                                                                                                                                                                                   |
| 5     | Ask Your Data — Analytics Chat                                  | `deepseek/deepseek-v4-pro` + `~deepseek/deepseek-v4-flash-latest` | A conversational layer on top of the Text-to-SQL workflow: multi-turn analysis, maintained query context (e.g. "Now break that down by city"), chart-ready results                                                                                                                                                                                                                                                          |
| 6     | Ticket Classification                                           | `nvidia/nemotron-3-nano-30b-a3b`                            | High-volume classification of support tickets by urgency, sentiment, and routing team — a good fit for a smaller, lower-cost model since it's classification, not open-ended generation                                                                                                                                                                                                                                     |
| 7     | Campaign Performance Narrative                                  | `~deepseek/deepseek-v4-flash-latest`                              | Converts the numeric output of a Text-to-SQL query into a concise, plain-English performance summary for non-technical stakeholders                                                                                                                                                                                                                                                                                         |
| 8     | Schema-Mapping Assistant — Client Onboarding                    | `z-ai/glm-5.3`                                              | RAG over a client's raw source columns and CDP schema documentation to propose field mappings, flagging ambiguous ones for human review instead of silently guessing                                                                                                                                                                                                                                                        |
| 9 | Persona Decision Loop — Customer Context & Next Best Action | `~deepseek/deepseek-v4-flash-latest`                              | Builds a grounded dynamic Persona from Customer 360 evidence, then derives Perception → Interpretation → Action. Combines behavioral events, campaign interactions, shopping-mall/offline experience, product comparisons, and customer history to determine the current customer context and recommend a measurable next-best action. The Persona is continuously updated when new customer outcomes become available. |


Every model string lives in one `MODELS` dict (see below) — swap any of them without touching the rest of the notebook.

### Requirements
- An **OpenRouter** API key (`OPENROUTER_API_KEY`)
- A free **Tavily** API key (`TAVILY_API_KEY`) — https://app.tavily.com — only needed for Task 1's web-search fallback; the rest of the notebook runs fine without it.
- In Google Colab: add both as Secrets (🔑 icon in the left sidebar). Outside Colab: set them as environment variables, or you'll be prompted interactively.

In [ ]:
# ============================================================
# 1. Install
# ============================================================

!pip -q install \
    pandas==2.2.3 \
    "opentelemetry-api>=1.39,<=1.42.1" \
    "opentelemetry-sdk>=1.39,<=1.42.1" \
    openai \
    langchain-openai \
    langchain-chroma \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-core \
    sentence-transformers \
    beautifulsoup4 \
    requests==2.32.4 \
    tavily-python

In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import json
import sqlite3

import requests
import bs4
import pandas as pd
import chromadb

from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from tavily import TavilyClient

In [ ]:
# ============================================================
# 3. API Keys (OpenRouter + Tavily)
#
# Works in Google Colab (Secrets) AND in plain Jupyter/local
# environments (env vars, with an interactive prompt as a
# last resort).
#
# In Colab:
#   Left sidebar -> 🔑 Secrets -> add OPENROUTER_API_KEY and
#   (optionally) TAVILY_API_KEY -> enable notebook access.
# ============================================================

def get_secret(name):
    """Fetch a secret from Colab userdata, then an env var, then an interactive prompt."""
    # 1) Google Colab secrets
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    # 2) Environment variable
    value = os.environ.get(name)
    if value:
        return value
    # 3) Interactive fallback (local Jupyter / secret not configured)
    try:
        import getpass
        value = getpass.getpass(f"Enter value for {name} (leave blank to skip): ")
        if value:
            return value
    except Exception:
        pass
    return None

# Accept either name so this still works with the original notebook's secret name
OPENROUTER_API_KEY = get_secret("OPENROUTER_API_KEY") or get_secret("OPEN_ROUTER_KEY")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "❌ OPENROUTER_API_KEY not found.\n\n"
        "In Google Colab: click the 🔑 Secrets icon -> add OPENROUTER_API_KEY -> enable notebook access.\n"
        "Elsewhere: set the OPENROUTER_API_KEY environment variable before running this cell."
    )

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ OpenRouter API key loaded")

if TAVILY_API_KEY:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    print("✅ Tavily API key loaded — Task 1's domain-restricted web fallback is enabled")
else:
    tavily_client = None
    print(
        "⚠️ TAVILY_API_KEY not found — Task 1's web-search fallback will report "
        "NOT_FOUND_ANSWERS instead of searching the web. "
        "Get a free key at https://app.tavily.com and add it as a secret to enable it."
    )

In [ ]:
# ============================================================
# 4. Model registry — one model per task, swappable in one place
# ============================================================

MODELS = {
    # --- 1-4: original use cases ---
    "customer_support":          "deepseek/deepseek-chat",           # cheap + grounded, high volume
    "text_to_sql":                "deepseek/deepseek-v4-pro",         # strongest coding/reasoning accuracy
    "marketing_orchestration":    "moonshotai/kimi-k3",               # best agentic tool-calling reliability
    "marketing_copy":             "~deepseek/deepseek-v4-flash-latest",     # cheap, fast, good enough for short copy

    # --- 5-8: additional use cases ---
    "analytics_chat_sql":         "deepseek/deepseek-v4-pro",         # multi-turn Text-to-SQL needs the same accuracy as Task 2
    "analytics_chat_narrative":   "~deepseek/deepseek-v4-flash-latest",     # cheap model just turning a table into a sentence
    "ticket_triage":              "nvidia/nemotron-3-nano-30b-a3b",   # cheapest tier -- classification, not generation, runs on every inbound ticket
    "campaign_narrative":         "~deepseek/deepseek-v4-flash-latest",     # cheap, fluent short-form prose from a results table
    "schema_mapping":             "z-ai/glm-5.3",                     # strong structured-reasoning model for column matching
    "analytics_persona":          "~deepseek/deepseek-v4-flash-latest"      # Persona Decision Loop Demo
}

def get_llm(task_key, temperature=0.1, max_tokens=1500):
    """Build a LangChain ChatOpenAI client pointed at OpenRouter for a given task."""
    model = MODELS[task_key]
    return ChatOpenAI(
        model=model,
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        temperature=temperature,
        max_tokens=max_tokens,
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com/",
            "X-Title": "LEO CDP - RAG Task Suite",
        },
    )

# Raw OpenAI-SDK client, used for the function-calling agent in Task 3
# (native tool-calling loop is simpler to control directly than through
# a LangChain wrapper for this use case).
raw_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

print("✅ Model registry ready:")
for k, v in MODELS.items():
    print(f"   {k:26s} -> {v}")

In [ ]:
# ============================================================
# 5. Quick sanity check — confirms the OpenRouter connection works
#    before we build any RAG pipelines
# ============================================================

_sanity = raw_client.chat.completions.create(
    model=MODELS["customer_support"],
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Explain LEO CDP in 3 simple sentences in Vietnamese."},
    ],
    temperature=0.7,
    max_tokens=300,
)
print(_sanity.choices[0].message.content)

In [ ]:
# ============================================================
# 6. Shared RAG building blocks (reused by every task below)
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# A single persistent Chroma client, shared across tasks, so that re-running
# a cell doesn't crash with a "collection already exists" error.
# Telemetry is disabled: chromadb's telemetry path is what pulls in
# opentelemetry, which is what conflicted with Colab's pinned versions
# during install -- now that we're creating many more collections (8
# use cases), it's worth switching off for good.
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
chroma_client = chromadb.Client(chromadb.config.Settings(anonymized_telemetry=False))

def load_url_as_document(url, user_agent="Mozilla/5.0 (compatible; LEO-CDP-RAG/1.0)"):
    """Download a web page and return its visible text as a single LangChain Document."""
    resp = requests.get(url, headers={"User-Agent": user_agent}, timeout=30)
    resp.raise_for_status()
    soup = bs4.BeautifulSoup(resp.text, "html.parser")
    for el in soup(["script", "style", "noscript", "template"]):
        el.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    return Document(page_content=text, metadata={"source": url})

def build_retriever(documents, collection_name, chunk_size=1000, chunk_overlap=200, k=3):
    """Split documents, embed them, and return a Chroma retriever.

    Safe to re-run: any existing collection with the same name is dropped first.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = splitter.split_documents(documents)

    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name=collection_name,
        client=chroma_client,
    )
    print(f"✅ '{collection_name}': {len(documents)} doc(s) -> {len(splits)} chunks indexed")
    return vectorstore.as_retriever(search_kwargs={"k": k})

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

def make_rag_chain(retriever, llm, prompt):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

def show(title, question, answer):
    bar = "=" * 80
    print(f"\n{bar}\n{title}\n{bar}")
    print(f"Q: {question}\n")
    print(f"A: {answer}")

def parse_json_safely(raw):
    """Best-effort JSON parsing for model output: strips stray markdown fences first.

    Returns a dict either way -- {'_raw': raw, '_parse_error': True} on failure --
    so downstream code never has to special-case a str vs. dict result.
    """
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"_raw": raw, "_parse_error": True}

## Task 1 — Customer Support

**Description:** RAG over a help-center knowledge base, with a Tavily domain-restricted web search fallback when the answer cannot be found in the KB.

**Flow:**
1. Try to answer from the indexed help-center KB.
2. If the KB says `NOT_FOUND_ANSWERS`, fall back to a **Tavily** web search restricted to `SUPPORT_DOMAIN` via `include_domains` — so the fallback only ever searches your own site, never the open web.
3. Re-answer using the search snippets as context, citing the source URL(s).

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `question` | `str` | `"What are the advantages of using RAG?"` | End-user's message, any language |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `answer` | `str` | `"RAG reduces hallucination by..."` | Markdown-formatted answer, or `NOT_FOUND_ANSWERS` if both KB and web fallback come up empty |
| `source` | `str` | `"internal KB"` or `"Tavily search (kbourne.github.io)"` | Which path produced the answer, useful for logging/QA |

In [ ]:
# ============================================================
# TASK 1 — Customer Support: build the KB retriever + strict RAG chain
# ============================================================

SUPPORT_URLS = [
    "https://kbourne.github.io/chapter1.html",
    "https://leocdp.com/",
    "https://www.bigdatavietnam.org/"
]
SUPPORT_DOMAIN = "kbourne.github.io"  # <-- point this at your real help-center domain in production

support_docs = [load_url_as_document(u) for u in SUPPORT_URLS]
support_retriever = build_retriever(support_docs, collection_name="support-kb", k=3)

support_prompt = ChatPromptTemplate.from_template("""
You are a strict AI customer-support assistant answering only from the provided Context.

CRITICAL RULES:
1. Base your answer ONLY on the Context. Never invent information or use outside knowledge.
2. Answer in the same language as the question. Use Markdown formatting when helpful.
3. STRICT FALLBACK: return exactly the string NOT_FOUND_ANSWERS and nothing else if the
   Context does not fully answer the question.

Context:
{context}

Question:
{question}

Answer:
""")

support_llm = get_llm("customer_support")
support_rag_chain = make_rag_chain(support_retriever, support_llm, support_prompt)
print("✅ Customer support RAG chain ready")

In [ ]:
# ============================================================
# TASK 1 — Tavily domain-restricted web-search fallback
# ============================================================

def tavily_domain_fallback(question, domain=SUPPORT_DOMAIN, max_results=5):
    """Search only within `domain` via Tavily, then answer using the search snippets as context."""
    if tavily_client is None:
        return "NOT_FOUND_ANSWERS (Tavily not configured — add TAVILY_API_KEY to enable this fallback)"

    results = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_domains=[domain],
        max_results=max_results,
    )
    hits = results.get("results", [])
    if not hits:
        return "NOT_FOUND_ANSWERS"

    web_context = "\n\n---\n\n".join(
        f"Source: {h['url']}\n{h.get('content', '')}" for h in hits
    )

    fallback_prompt = ChatPromptTemplate.from_template("""
You are a customer-support assistant. Answer the question using ONLY the web search
results below (all restricted to {domain}). Cite the source URL(s) you used at the end.
If the results still don't answer the question, reply exactly: NOT_FOUND_ANSWERS

Search results:
{web_context}

Question:
{question}

Answer:
""")
    chain = fallback_prompt | support_llm | StrOutputParser()
    return chain.invoke({"web_context": web_context, "question": question, "domain": domain})


def answer_customer_support(question):
    """RAG first; if the KB doesn't cover it, fall back to a domain-restricted live web search."""
    answer = support_rag_chain.invoke(question)
    source = "internal KB"
    if answer.strip() == "NOT_FOUND_ANSWERS":
        answer = tavily_domain_fallback(question)
        source = f"Tavily search ({SUPPORT_DOMAIN})"
    return answer, source

print("✅ Tavily fallback wired up")

In [ ]:
# --- Test 1a: answer should come straight from the indexed KB ---
q1 = "What are the advantages of using RAG?"
a1, src1 = answer_customer_support(q1)
show(f"TASK 1 · Customer Support  [source: {src1}]", q1, a1)

# --- Test 1b: KB is silent on this -> should trigger the Tavily domain-restricted fallback ---
q2 = "Does kbourne.github.io have a chapter comparing fine-tuning vs RAG trade-offs?"
a2, src2 = answer_customer_support(q2)
show(f"TASK 1 · Customer Support  [source: {src2}]", q2, a2)

## Task 2 — Text-to-SQL

**Description:** Schema-grounded RAG → SQL generation → SQL execution against a small sample CDP database.

**Flow:**
1. A sample "LEO CDP" SQLite database is created (`customers`, `segments`, `segment_members`, `campaigns`, `campaign_sends`, `events`, `support_tickets`).
2. Table schemas are stored as retrievable Documents — RAG pulls in only the tables relevant to the question, so the prompt scales to a real schema with hundreds of tables instead of pasting the whole schema into every call.
3. The model outputs **SQL only** (SELECT-only, enforced), which is then executed against the sample DB and returned as a DataFrame.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `question` | `str` | `"How many customers are in the VIP segment?"` | Natural-language analytics question |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `sql` | `str` | `"SELECT COUNT(*) ..."` | The generated SELECT statement, or `NOT_FOUND_ANSWERS` if the schema can't answer it |
| `result` | `pandas.DataFrame` \| `str` \| `None` | a DataFrame of rows | `None` if `sql == NOT_FOUND_ANSWERS`; a string starting with `⚠️` if execution failed |

In [ ]:
# ============================================================
# TASK 2 — Text-to-SQL: build a small sample "LEO CDP" database
#
# Expanded beyond the original 4-use-case version to also back
# Tasks 5-8: `events` (behavioral/purchase history, available for
# richer Text-to-SQL and Ask Your Data questions), `support_tickets`
# (Ticket Classification), and a second campaign + its sends (a
# same-time-last-year comparison for the Campaign Performance
# Narrative task).
# ============================================================

sql_conn = sqlite3.connect(":memory:")

pd.DataFrame([
    (1, "Nguyen Van A", "a@example.com", "Ho Chi Minh City", "2024-01-15", 4200000),
    (2, "Tran Thi B",   "b@example.com", "Hanoi",            "2024-03-02", 950000),
    (3, "Le Van C",     "c@example.com", "Da Nang",          "2023-11-20", 12800000),
    (4, "Pham Thi D",   "d@example.com", "Ho Chi Minh City", "2025-02-10", 300000),
    (5, "Hoang Van E",  "e@example.com", "Can Tho",          "2024-07-05", 7600000),
], columns=["customer_id", "full_name", "email", "city", "signup_date", "ltv"]
).to_sql("customers", sql_conn, index=False, if_exists="replace")

pd.DataFrame([
    (1, "VIP", "customers with ltv > 5,000,000 VND"),
    (2, "New Signups 2025", "customers who signed up in 2025"),
], columns=["segment_id", "segment_name", "definition"]
).to_sql("segments", sql_conn, index=False, if_exists="replace")

pd.DataFrame([
    (1, 3), (1, 5), (2, 4),
], columns=["segment_id", "customer_id"]
).to_sql("segment_members", sql_conn, index=False, if_exists="replace")

# --- campaigns: Tet2026 (current) + New Year push + Tet2025 (for Task 10's
#     year-over-year comparison) ---
pd.DataFrame([
    (1, "Tet2026 Email Blast", "email", "2026-01-20", "sent"),
    (2, "New Year Push Notif", "push",  "2026-01-01", "sent"),
    (3, "Tet2025 Email Blast", "email", "2025-01-25", "sent"),
], columns=["campaign_id", "campaign_name", "channel", "start_date", "status"]
).to_sql("campaigns", sql_conn, index=False, if_exists="replace")

pd.DataFrame([
    (1, 1, 1, "2026-01-20 09:00", 1, 1),
    (2, 1, 3, "2026-01-20 09:00", 1, 0),
    (3, 1, 5, "2026-01-20 09:00", 0, 0),
    (4, 2, 4, "2026-01-01 08:00", 1, 1),
    # Tet2025 sends to the same VIP-ish recipients, for a year-over-year comparison
    (5, 3, 1, "2025-01-25 09:00", 1, 0),
    (6, 3, 3, "2025-01-25 09:00", 0, 0),
    (7, 3, 5, "2025-01-25 09:00", 0, 0),
], columns=["send_id", "campaign_id", "customer_id", "sent_at", "opened", "clicked"]
).to_sql("campaign_sends", sql_conn, index=False, if_exists="replace")

# --- events: behavioral/purchase history, available for richer
#     Text-to-SQL (Task 2) and Ask Your Data (Task 5) questions ---
pd.DataFrame([
    # Customer 1 -- recently active
    (1,  1, "app_open",       "2026-02-14 08:00", "app", 0),
    (2,  1, "page_view",      "2026-02-14 08:05", "app", 0),
    (3,  1, "purchase",       "2026-02-10 19:30", "web", 1500000),
    (4,  1, "page_view",      "2026-01-20 10:00", "web", 0),
    # Customer 2 -- went quiet in early January
    (5,  2, "app_open",       "2026-01-05 07:00", "app", 0),
    (6,  2, "cart_abandon",   "2026-01-05 07:20", "app", 0),
    (7,  2, "purchase",       "2025-12-01 15:00", "web", 400000),
    # Customer 3 -- VIP, long inactive since late November
    (8,  3, "purchase",       "2025-11-25 12:00", "web", 3000000),
    (9,  3, "page_view",      "2025-11-20 09:00", "web", 0),
    (10, 3, "purchase",       "2025-09-10 12:00", "web", 5000000),
    # Customer 4 -- new signup, recently active
    (11, 4, "app_open",       "2026-02-08 09:00", "app", 0),
    (12, 4, "page_view",      "2026-02-08 09:05", "app", 0),
    (13, 4, "cart_abandon",   "2026-02-08 09:20", "app", 0),
    # Customer 5 -- VIP, inactive since late December
    (14, 5, "purchase",       "2025-12-20 14:00", "web", 2200000),
    (15, 5, "support_contact","2025-12-22 10:00", "web", 0),
    (16, 5, "page_view",      "2025-12-10 08:00", "web", 0),
], columns=["event_id", "customer_id", "event_name", "event_time", "channel", "value"]
).to_sql("events", sql_conn, index=False, if_exists="replace")

# --- support_tickets: used by Task 6 (Ticket Triage) ---
pd.DataFrame([
    (1, 3,    "email",           "Payment failed for premium renewal",
     "My premium subscription renewal failed twice today and I was charged anyway. "
     "This is urgent, I need this fixed before my trip tomorrow.", "2026-02-14 09:15"),
    (2, 4,    "chat",            "Cannot log in after password reset",
     "I reset my password following the email link but the app still says invalid "
     "credentials. Tried three times.", "2026-02-13 14:02"),
    (3, 1,    "email",           "Question about loyalty points expiry",
     "Hi, just wondering when my loyalty points expire? Not urgent, just curious.",
     "2026-02-12 11:30"),
    (4, 2,    "email",           "Really unhappy with the app update",
     "The new update is terrible, it crashes every time I open the cart. Extremely "
     "frustrating, I am considering cancelling.", "2026-02-11 20:45"),
    (5, 5,    "phone_followup",  "Refund request for duplicate charge",
     "I was charged twice for the same order last week. Please refund the duplicate "
     "charge as soon as possible.", "2026-02-10 16:20"),
    (6, None, "web_form",        "Feature request: dark mode",
     "Would love to see a dark mode option in the app. Not a big deal, just a "
     "nice-to-have.", "2026-02-09 10:00"),
    (7, 1,    "email",           "Great experience with support team!",
     "Just wanted to say thank you, your support agent Lan helped me resolve my "
     "issue quickly and was very kind.", "2026-02-08 13:10"),
], columns=["ticket_id", "customer_id", "channel", "subject", "message", "created_at"]
).to_sql("support_tickets", sql_conn, index=False, if_exists="replace")

print(
    "✅ Sample LEO CDP SQLite DB created: customers, segments, segment_members, "
    "campaigns, campaign_sends, events, support_tickets"
)

In [ ]:
# ============================================================
# TASK 2 — Schema documentation as retrievable context
# ============================================================

schema_docs = [
    Document(page_content=(
        "Table: customers\n"
        "Columns: customer_id (INTEGER, PK), full_name (TEXT), email (TEXT), "
        "city (TEXT), signup_date (TEXT, YYYY-MM-DD), ltv (INTEGER, lifetime value in VND)\n"
        "Description: one row per customer profile in the CDP."
    ), metadata={"table": "customers"}),
    Document(page_content=(
        "Table: segments\n"
        "Columns: segment_id (INTEGER, PK), segment_name (TEXT), definition (TEXT)\n"
        "Description: named customer segments (e.g. VIP)."
    ), metadata={"table": "segments"}),
    Document(page_content=(
        "Table: segment_members\n"
        "Columns: segment_id (INTEGER, FK -> segments.segment_id), "
        "customer_id (INTEGER, FK -> customers.customer_id)\n"
        "Description: many-to-many mapping of which customers belong to which segment."
    ), metadata={"table": "segment_members"}),
    Document(page_content=(
        "Table: campaigns\n"
        "Columns: campaign_id (INTEGER, PK), campaign_name (TEXT), channel (TEXT: email/push/sms), "
        "start_date (TEXT), status (TEXT)\n"
        "Description: marketing campaigns run from the CDP."
    ), metadata={"table": "campaigns"}),
    Document(page_content=(
        "Table: campaign_sends\n"
        "Columns: send_id (INTEGER, PK), campaign_id (INTEGER, FK -> campaigns.campaign_id), "
        "customer_id (INTEGER, FK -> customers.customer_id), sent_at (TEXT), "
        "opened (INTEGER 0/1), clicked (INTEGER 0/1)\n"
        "Description: one row per message sent to one customer in one campaign."
    ), metadata={"table": "campaign_sends"}),
    Document(page_content=(
        "Table: events\n"
        "Columns: event_id (INTEGER, PK), customer_id (INTEGER, FK -> customers.customer_id), "
        "event_name (TEXT: app_open/page_view/purchase/cart_abandon/support_contact), "
        "event_time (TEXT, YYYY-MM-DD HH:MM), channel (TEXT: web/app), "
        "value (INTEGER, VND amount for purchase events, 0 otherwise)\n"
        "Description: one row per behavioral/transactional event for a customer -- the "
        "raw activity feed used to compute recency, engagement, and purchase history."
    ), metadata={"table": "events"}),
    Document(page_content=(
        "Table: support_tickets\n"
        "Columns: ticket_id (INTEGER, PK), customer_id (INTEGER, FK -> customers.customer_id, "
        "NULLable for tickets from non-customers/prospects), channel (TEXT: email/chat/"
        "phone_followup/web_form), subject (TEXT), message (TEXT, the customer's own words), "
        "created_at (TEXT, YYYY-MM-DD HH:MM)\n"
        "Description: one row per inbound support ticket, before triage/classification."
    ), metadata={"table": "support_tickets"}),
]

schema_retriever = build_retriever(
    schema_docs, collection_name="cdp-schema", chunk_size=500, chunk_overlap=0, k=5
)

In [ ]:
# ============================================================
# TASK 2 — Text-to-SQL prompt, generation chain, and safe execution
# ============================================================

sql_prompt = ChatPromptTemplate.from_template("""
You are a Text-to-SQL engine for a SQLite database. Use ONLY the tables/columns in the
Context below — never invent tables or columns.

RULES:
1. Output ONLY a single valid SQLite SELECT statement. No explanation, no markdown fences.
2. Never write INSERT/UPDATE/DELETE/DROP/ALTER — read-only queries only.
3. If the question cannot be answered with the given schema, output exactly: NOT_FOUND_ANSWERS

Context (schema):
{context}

Question:
{question}

SQL:
""")

sql_llm = get_llm("text_to_sql")
text_to_sql_chain = make_rag_chain(schema_retriever, sql_llm, sql_prompt)

FORBIDDEN_KEYWORDS = ("insert", "update", "delete", "drop", "alter", "attach", "pragma")

def run_text_to_sql(question):
    """Generate SQL from a natural-language question, then safely execute it."""
    raw = text_to_sql_chain.invoke(question).strip()
    # Strip accidental markdown fences if the model adds them anyway
    sql = raw.strip("`")
    if sql.lower().startswith("sql"):
        sql = sql[3:].strip()

    if sql == "NOT_FOUND_ANSWERS":
        return sql, None

    if any(word in sql.lower() for word in FORBIDDEN_KEYWORDS):
        raise ValueError(f"Refusing to execute a non-SELECT statement: {sql}")

    try:
        df = pd.read_sql_query(sql, sql_conn)
    except Exception as e:
        return sql, f"⚠️ SQL execution error: {e}"
    return sql, df

print("✅ Text-to-SQL chain ready")

In [ ]:
# --- Test 2a: aggregate count via a join ---
q3 = "How many customers are in the VIP segment?"
sql3, result3 = run_text_to_sql(q3)
show("TASK 2 · Text-to-SQL", q3, f"SQL: {sql3}")
if isinstance(result3, pd.DataFrame):
    display(result3)
else:
    print(result3)

# --- Test 2b: multi-table join ---
q4 = "List the full name and email of customers who opened the Tet2026 Email Blast campaign"
sql4, result4 = run_text_to_sql(q4)
show("TASK 2 · Text-to-SQL", q4, f"SQL: {sql4}")
if isinstance(result4, pd.DataFrame):
    display(result4)
else:
    print(result4)

## Task 3 — Marketing Automation: Campaign Orchestration

**Description:** A RAG-grounded, tool-calling agent that plans and executes a multi-step campaign workflow.

**Flow:**
1. An internal "campaign launch playbook" is indexed and retrieved as grounding context.
2. The model is given four tools (`get_segment_size`, `check_frequency_cap`, `send_test_message`, `schedule_campaign`) that it must call **in the order the playbook requires**.
3. A small agent loop executes each tool call against the sample CDP database from Task 2, feeds the result back to the model, and repeats until the model returns a final plain-text summary.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `brief` | `str` | `"Launch the 'Tet2026 Email Blast' campaign to the VIP segment, sending at 2026-01-27 09:00."` | Natural-language campaign launch instruction |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `summary` | `str` | `"Segment size 2 (>=3 required)..."` | Final plain-text summary of every check performed and its result |
| `trace` | `list[dict]` | full message history | Every system/user/assistant/tool message, useful for auditing what the agent actually did |

In [ ]:
# ============================================================
# TASK 3 — Marketing Automation (Campaign Orchestration): playbook as grounding context
# ============================================================

playbook_text = """
LEO CDP Campaign Launch Playbook

Rule 1: Never launch a campaign to a segment smaller than 3 customers -- too small to be
statistically meaningful. Check segment size first with get_segment_size.

Rule 2: Always send a test message to a single internal test recipient BEFORE scheduling
the full send. Use send_test_message.

Rule 3: Respect the frequency cap -- a customer must not receive more than 1 campaign
message per 7 days. Use check_frequency_cap before scheduling.

Rule 4: Only schedule a campaign after Rules 1-3 have all passed. Use schedule_campaign
as the final step, and report a clear summary of every check performed.
"""

playbook_doc = Document(page_content=playbook_text, metadata={"source": "internal-playbook"})
playbook_retriever = build_retriever(
    [playbook_doc], collection_name="campaign-playbook", chunk_size=400, chunk_overlap=0, k=4
)

playbook_context = format_docs(playbook_retriever.invoke("campaign launch rules"))
print("✅ Playbook indexed and retrieved as orchestration context")

In [ ]:
# ============================================================
# TASK 3 — Mock campaign tools (query the Task 2 sample DB where relevant)
# ============================================================

def get_segment_size(segment_name):
    df = pd.read_sql_query(f"""
        SELECT COUNT(*) AS n FROM segment_members sm
        JOIN segments s ON s.segment_id = sm.segment_id
        WHERE s.segment_name = '{segment_name}'
    """, sql_conn)
    return {"segment_name": segment_name, "size": int(df["n"].iloc[0])}

def check_frequency_cap(segment_name):
    # Demo logic: count sends to this segment's members in a trailing 7-day window
    df = pd.read_sql_query(f"""
        SELECT COUNT(*) AS recent_sends FROM campaign_sends cs
        JOIN segment_members sm ON sm.customer_id = cs.customer_id
        JOIN segments s ON s.segment_id = sm.segment_id
        WHERE s.segment_name = '{segment_name}'
          AND date(cs.sent_at) >= date('2026-01-20', '-7 days')
    """, sql_conn)
    recent = int(df["recent_sends"].iloc[0])
    return {"segment_name": segment_name, "recent_sends_last_7_days": recent, "cap_ok": recent < 5}

def send_test_message(campaign_name, test_recipient="qa-team@internal.leo-cdp.com"):
    return {"campaign_name": campaign_name, "test_recipient": test_recipient, "status": "test_sent_ok"}

def schedule_campaign(campaign_name, segment_name, send_time):
    return {
        "campaign_name": campaign_name,
        "segment_name": segment_name,
        "send_time": send_time,
        "status": "scheduled",
    }

AVAILABLE_TOOLS = {
    "get_segment_size": get_segment_size,
    "check_frequency_cap": check_frequency_cap,
    "send_test_message": send_test_message,
    "schedule_campaign": schedule_campaign,
}

TOOL_SCHEMA = [
    {"type": "function", "function": {
        "name": "get_segment_size",
        "description": "Return how many customers belong to a named segment.",
        "parameters": {
            "type": "object",
            "properties": {"segment_name": {"type": "string"}},
            "required": ["segment_name"],
        },
    }},
    {"type": "function", "function": {
        "name": "check_frequency_cap",
        "description": "Check whether a segment is within the 1-message-per-7-days frequency cap.",
        "parameters": {
            "type": "object",
            "properties": {"segment_name": {"type": "string"}},
            "required": ["segment_name"],
        },
    }},
    {"type": "function", "function": {
        "name": "send_test_message",
        "description": "Send a single test message for a campaign to the internal QA recipient.",
        "parameters": {
            "type": "object",
            "properties": {"campaign_name": {"type": "string"}},
            "required": ["campaign_name"],
        },
    }},
    {"type": "function", "function": {
        "name": "schedule_campaign",
        "description": "Schedule the full campaign send after all checks pass.",
        "parameters": {
            "type": "object",
            "properties": {
                "campaign_name": {"type": "string"},
                "segment_name": {"type": "string"},
                "send_time": {"type": "string"},
            },
            "required": ["campaign_name", "segment_name", "send_time"],
        },
    }},
]

print(f"✅ {len(AVAILABLE_TOOLS)} campaign tools registered")

In [ ]:
# ============================================================
# TASK 3 — RAG-grounded tool-calling agent loop
# ============================================================

def run_marketing_orchestration(brief, max_turns=8):
    """Run a playbook-grounded, tool-calling agent to plan and 'execute' a campaign launch."""
    messages = [
        {"role": "system", "content": (
            "You are a marketing-operations agent for LEO CDP. Follow the playbook rules "
            "below exactly, in order, using the provided tools. Do not skip a rule. "
            "When finished, reply with a short plain-text summary of every check you ran "
            "and its result -- no more tool calls after that.\n\n"
            f"PLAYBOOK CONTEXT:\n{playbook_context}"
        )},
        {"role": "user", "content": brief},
    ]

    for _ in range(max_turns):
        resp = raw_client.chat.completions.create(
            model=MODELS["marketing_orchestration"],
            messages=messages,
            tools=TOOL_SCHEMA,
            temperature=0.1,
            max_tokens=800,
        )
        msg = resp.choices[0].message
        tool_calls = getattr(msg, "tool_calls", None)

        if not tool_calls:
            return msg.content, messages

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in tool_calls],
        })

        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments or "{}")
            print(f"🔧 tool call: {fn_name}({fn_args})")
            result = AVAILABLE_TOOLS[fn_name](**fn_args)
            print(f"   -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return "⚠️ Max turns reached without a final answer.", messages

print("✅ Orchestration agent ready")

In [ ]:
brief = "Launch the 'Tet2026 Email Blast' campaign to the VIP segment, sending at 2026-01-27 09:00."
summary, trace = run_marketing_orchestration(brief)
show("TASK 3 · Marketing Automation (Orchestration)", brief, summary)

## Task 4 — Marketing Automation: Copy Drafting

**Description:** Brand-guideline-grounded structured copy generation, including subject line, preview text, email body, and social caption.

**Flow:**
1. Brand voice guidelines are indexed as retrievable context.
2. Given a campaign brief, the model returns **structured JSON** (subject line, preview text, body, social caption) so it can be piped straight into an ESP/campaign tool downstream.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `brief` | `str` | `"Announce our new Tet 2026 loyalty bonus..."` | Natural-language campaign brief |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `subject_line` | `str` | `"🧧 2x points all Tet week"` | Max 60 chars, 1 emoji allowed |
| `preview_text` | `str` | `"Your VIP bonus starts now"` | Max 90 chars |
| `body` | `str` | 2–3 sentences | The email body; follows brand tone rules |
| `social_caption` | `str` | 1 sentence | No emoji |

In [ ]:
# ============================================================
# TASK 4 — Marketing Automation (Copy Drafting)
# ============================================================

brand_guidelines_text = """
LEO CDP Brand Voice Guidelines

Tone: warm, confident, plain-spoken. Avoid corporate jargon like "synergy" or "leverage".
Always mention the customer's benefit before the product feature.
Never use ALL CAPS for emphasis; use bold Markdown instead.
Emojis are allowed sparingly (max 1 per message) for email subject lines only.
Sign-off for all customer emails: "-- The LEO CDP Team".
"""

brand_doc = Document(page_content=brand_guidelines_text, metadata={"source": "brand-guidelines"})
brand_retriever = build_retriever(
    [brand_doc], collection_name="brand-guidelines", chunk_size=400, chunk_overlap=0, k=4
)

copy_prompt = ChatPromptTemplate.from_template("""
You are a marketing copywriter for LEO CDP. Follow the brand Context below strictly.

Write output as JSON with exactly these keys:
- "subject_line": string (max 60 characters, 1 emoji allowed)
- "preview_text": string (max 90 characters)
- "body": string (2-3 short sentences)
- "social_caption": string (1 sentence, no emoji)

Return ONLY the JSON object, no markdown fences, no extra text.

Brand Context:
{context}

Campaign brief:
{question}

JSON:
""")

copy_llm = get_llm("marketing_copy")
copy_chain = make_rag_chain(brand_retriever, copy_llm, copy_prompt)
print("✅ Copy-drafting RAG chain ready")

In [ ]:
brief4 = (
    "Announce our new Tet 2026 loyalty bonus: VIP customers get 2x reward points "
    "on all purchases during Tet week."
)
raw_json = copy_chain.invoke(brief4)

try:
    copy_result = json.loads(raw_json)
    pretty = json.dumps(copy_result, indent=2, ensure_ascii=False)
except json.JSONDecodeError:
    pretty = raw_json  # show raw text if the model didn't return clean JSON

show("TASK 4 · Marketing Automation (Copy Drafting)", brief4, pretty)

## Task 5 — Ask Your Data: Analytics Chat

**Description:** A conversational layer on top of the Text-to-SQL workflow. It supports multi-turn analysis, maintains query context (e.g. "Now break that down by city"), and returns chart-ready results rather than a one-shot SQL response.

**Flow:**
1. Reuses Task 2's `schema_retriever` to pull in only the relevant table schemas for the new question.
2. Builds a short conversation-history block (previous question → SQL → row count) and includes it in the SQL-generation prompt, so the model can resolve references like "that" or "the same segment".
3. Generates SQL (same SELECT-only guard as Task 2) and executes it.
4. A second, cheap model call turns the resulting table into a 1–2 sentence narrative and proposes one natural follow-up question.
5. The turn (question, SQL, row count) is appended to the session's history for the next call.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `question` | `str` | `"Now break that down by city"` | Can reference prior turns; session state lives in an `AnalyticsChatSession` instance |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `sql` | `str` | `"SELECT city, COUNT(*) ..."` | Generated SQL for this turn |
| `result` | `pandas.DataFrame` \| `None` | table of rows | Chart-ready: pass straight to a plotting call; `None` if the schema can't answer the question |
| `narrative` | `str` | `"Most VIP customers are in Da Nang and Can Tho."` | Plain-English summary of the result |
| `suggested_followup` | `str` | `"Want to see this by signup year instead?"` | One natural next question |

In [ ]:
# ============================================================
# TASK 5 — "Ask Your Data" Analytics Chat
#
# Reuses Task 2's `schema_retriever`, `sql_conn`, and `FORBIDDEN_KEYWORDS`.
# Adds: conversation history in the SQL prompt, and a second cheap-model
# call that turns the result table into a narrative + follow-up question.
# ============================================================

analytics_sql_prompt = ChatPromptTemplate.from_template("""
You are a Text-to-SQL engine for a SQLite database, running inside a multi-turn chat.
Use ONLY the tables/columns in the Schema Context below -- never invent tables or columns.
If the question refers back to a previous turn (e.g. "that", "the same segment", "now
break it down by..."), use the Conversation History to resolve what it refers to.

RULES:
1. Output ONLY a single valid SQLite SELECT statement. No explanation, no markdown fences.
2. Never write INSERT/UPDATE/DELETE/DROP/ALTER -- read-only queries only.
3. If the question cannot be answered with the given schema, output exactly: NOT_FOUND_ANSWERS

Conversation History (most recent last):
{history}

Schema Context:
{context}

New question:
{question}

SQL:
""")

analytics_narrative_prompt = ChatPromptTemplate.from_template("""
You are a data analyst summarizing a query result for a non-technical stakeholder.

Given the question and the resulting table (as CSV), return ONLY a JSON object with:
- "narrative": a 1-2 sentence plain-English summary of the key finding
- "suggested_followup": one natural, specific follow-up question the user might ask next

Question: {question}

Result table (CSV):
{table_csv}

JSON:
""")

analytics_sql_llm = get_llm("analytics_chat_sql")
analytics_narrative_llm = get_llm("analytics_chat_narrative")

analytics_sql_chain = analytics_sql_prompt | analytics_sql_llm | StrOutputParser()
analytics_narrative_chain = analytics_narrative_prompt | analytics_narrative_llm | StrOutputParser()


class AnalyticsChatSession:
    """Keeps just enough history (question, SQL, row count) to resolve follow-up questions."""

    def __init__(self):
        self.history = []

    def _history_block(self):
        if not self.history:
            return "(no previous turns)"
        lines = [
            f"{i}. Q: {t['question']}\n   SQL: {t['sql']}\n   Rows returned: {t['row_count']}"
            for i, t in enumerate(self.history, 1)
        ]
        return "\n".join(lines)

    def ask(self, question):
        context = format_docs(schema_retriever.invoke(question))
        history_block = self._history_block()

        raw_sql = analytics_sql_chain.invoke(
            {"context": context, "history": history_block, "question": question}
        ).strip()
        sql = raw_sql.strip("`")
        if sql.lower().startswith("sql"):
            sql = sql[3:].strip()

        if sql == "NOT_FOUND_ANSWERS":
            self.history.append({"question": question, "sql": sql, "row_count": 0})
            return {
                "sql": sql, "result": None,
                "narrative": "I couldn't find data to answer that with the current schema.",
                "suggested_followup": None,
            }

        if any(word in sql.lower() for word in FORBIDDEN_KEYWORDS):
            raise ValueError(f"Refusing to execute a non-SELECT statement: {sql}")

        df = pd.read_sql_query(sql, sql_conn)
        self.history.append({"question": question, "sql": sql, "row_count": len(df)})

        raw_narrative = analytics_narrative_chain.invoke(
            {"question": question, "table_csv": df.to_csv(index=False)}
        )
        parsed = parse_json_safely(raw_narrative)
        narrative = parsed.get("narrative") or (raw_narrative if parsed.get("_parse_error") else "")
        suggested_followup = parsed.get("suggested_followup")

        return {"sql": sql, "result": df, "narrative": narrative, "suggested_followup": suggested_followup}


print("✅ Analytics chat session ready")

In [ ]:
session = AnalyticsChatSession()

# --- Turn 1 ---
turn1 = session.ask("How many customers are in the VIP segment?")
show("TASK 5 · Ask Your Data (turn 1)", "How many customers are in the VIP segment?", f"SQL: {turn1['sql']}")
if isinstance(turn1["result"], pd.DataFrame):
    display(turn1["result"])
print(f"📝 Narrative: {turn1['narrative']}")
print(f"➡️  Suggested follow-up: {turn1['suggested_followup']}")

# --- Turn 2: a follow-up that only makes sense with turn 1's context ---
turn2 = session.ask("Now break that down by city instead")
show("TASK 5 · Ask Your Data (turn 2)", "Now break that down by city instead", f"SQL: {turn2['sql']}")
if isinstance(turn2["result"], pd.DataFrame):
    display(turn2["result"])
print(f"📝 Narrative: {turn2['narrative']}")
print(f"➡️  Suggested follow-up: {turn2['suggested_followup']}")

## Task 6 — Ticket Classification

**Description:** High-volume classification of support tickets by urgency, sentiment, and routing team. This workload is a good candidate for a smaller, lower-cost model because it requires classification rather than open-ended generation — classification here is grounded in an internal routing/priority policy document rather than the model's own unguided judgment, and a VIP customer (looked up from the CDP database) gets a priority bump, same as a real routing system would apply.

**Flow:**
1. Retrieve the routing/priority policy context relevant to each ticket (RAG over an internal policy doc).
2. Look up the ticket's customer in the CDP database (if any) to check VIP status.
3. Classify the ticket into structured JSON using the cheap triage model.
4. Batch-process all sample tickets and display the results as a DataFrame for quick scanning.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `ticket_id` | `int` | `1` | Looked up from `support_tickets`; `customer_id` may be `NULL` for prospects |
| `subject` + `message` | `str` | `"Payment failed..."` | The ticket's own text, used for classification |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `category` | `str` | `"Billing"` | One of a fixed set of categories (see policy doc) |
| `priority` | `str` | `"P1"` | `P1` (urgent) to `P4` (low) — this is the ticket's urgency level; VIP customers get bumped one level |
| `sentiment` | `str` | `"negative"` | `positive` / `neutral` / `negative` |
| `routing_team` | `str` | `"Billing/Payments"` | Which internal team should own the ticket |
| `one_line_summary` | `str` | `"VIP customer double-charged, needs urgent refund."` | For a triage queue's list view |

In [ ]:
# ============================================================
# TASK 6 — Ticket Classification
# ============================================================

routing_policy_text = """
LEO CDP Support Ticket Routing & Priority Policy

Categories: Billing, Account/Login, Bug Report, Feature Request, General Inquiry, Praise/Feedback.

Priority levels:
- P1 (urgent): billing/payment failures, security issues, or anything blocking the
  customer from using a paid feature right now.
- P2 (high): bugs that break a core workflow, or any negative-sentiment ticket from a
  VIP customer.
- P3 (medium): login/account issues that have a workaround, general inquiries with a
  time-sensitive element.
- P4 (low): feature requests, praise/feedback, and non-urgent general questions.

VIP customers get their computed priority bumped one level higher (e.g. P3 -> P2),
except tickets already at P1.

Routing teams:
- Billing/Payments: category = Billing
- Technical Support: category = Account/Login
- Engineering: category = Bug Report
- Product: category = Feature Request
- Customer Success: category = General Inquiry or Praise/Feedback
"""

routing_policy_doc = Document(page_content=routing_policy_text, metadata={"source": "routing-policy"})
routing_policy_retriever = build_retriever(
    [routing_policy_doc], collection_name="ticket-routing-policy", chunk_size=500, chunk_overlap=0, k=4
)

def get_customer_vip_status(customer_id):
    """True if the (possibly NULL, possibly NaN) customer_id belongs to the VIP segment."""
    if customer_id is None or (isinstance(customer_id, float) and pd.isna(customer_id)):
        return False
    df = pd.read_sql_query(f"""
        SELECT COUNT(*) AS n FROM segment_members sm
        JOIN segments s ON s.segment_id = sm.segment_id
        WHERE s.segment_name = 'VIP' AND sm.customer_id = {int(customer_id)}
    """, sql_conn)
    return int(df["n"].iloc[0]) > 0

triage_prompt = ChatPromptTemplate.from_template("""
You are a support-ticket triage classifier. Use ONLY the routing policy Context below.

Return ONLY a JSON object with exactly these keys:
- "category": one of Billing, Account/Login, Bug Report, Feature Request, General Inquiry, Praise/Feedback
- "priority": one of P1, P2, P3, P4 (apply the VIP bump rule if is_vip is true)
- "sentiment": one of positive, neutral, negative
- "routing_team": the team per the policy's category-to-team mapping
- "one_line_summary": a single sentence summarizing the ticket for a triage queue

Routing Policy Context:
{context}

Ticket:
Subject: {subject}
Message: {message}
Customer is VIP: {is_vip}

JSON:
""")

triage_llm = get_llm("ticket_triage")
triage_chain = triage_prompt | triage_llm | StrOutputParser()

def triage_ticket(ticket_row):
    """Classify one support_tickets row (a pandas Series) into structured triage fields."""
    is_vip = get_customer_vip_status(ticket_row["customer_id"])
    context = format_docs(
        routing_policy_retriever.invoke(f"{ticket_row['subject']} {ticket_row['message']}")
    )
    raw = triage_chain.invoke({
        "context": context,
        "subject": ticket_row["subject"],
        "message": ticket_row["message"],
        "is_vip": is_vip,
    })
    result = parse_json_safely(raw)
    result["ticket_id"] = ticket_row["ticket_id"]
    result["is_vip"] = is_vip
    return result

print("✅ Ticket triage classifier ready")

In [ ]:
tickets_df = pd.read_sql_query("SELECT * FROM support_tickets", sql_conn)
triage_results = [triage_ticket(row) for _, row in tickets_df.iterrows()]

triage_df = pd.DataFrame(triage_results)
cols = ["ticket_id", "is_vip", "category", "priority", "sentiment", "routing_team", "one_line_summary"]
triage_df = triage_df[[c for c in cols if c in triage_df.columns]]

print(f"\n{'=' * 80}\nTASK 6 · Ticket Classification — {len(triage_df)} tickets classified\n{'=' * 80}")
display(triage_df)

## Task 7 — Campaign Performance Narrative

**Description:** Converts the numeric output of a Text-to-SQL query into a concise, plain-English performance summary for non-technical stakeholders (e.g. "Open rate dropped 12% versus the previous campaign, mainly driven by the Hanoi segment.").

**Flow:**
1. Ask a natural-language comparison question (e.g. this year vs. last year), answered by Task 2's existing schema-grounded Text-to-SQL chain (`run_text_to_sql`).
2. Execute the SQL and get a small results table (per-campaign send count, open rate, click rate).
3. A cheap model turns that table into a short narrative highlighting the notable deltas, plus a structured `key_metrics` dict for downstream dashboarding.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `question` | `str` | `"Compare open rate and click rate between Tet2026 Email Blast and Tet2025 Email Blast"` | Same free-form style as Task 2's questions |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `narrative` | `str` | `"Tet2026 nearly doubled last year's open rate..."` | 3–5 sentence plain-English summary; direction (up or down) follows whatever the data actually shows |
| `key_metrics` | `dict` | `{"Tet2026 Email Blast": {"open_rate": 0.67, ...}}` | Extracted straight from the result table, for a dashboard to consume without re-parsing prose |

In [ ]:
# ============================================================
# TASK 7 — Campaign Performance Narrative
#
# Composes with Task 2 directly: `run_text_to_sql` produces the table,
# this task's only new piece is turning that table into prose + a
# generic key_metrics dict (keyed by whatever the first column is,
# so it works regardless of the exact column names the SQL model chose).
# ============================================================

campaign_narrative_prompt = ChatPromptTemplate.from_template("""
You are a marketing analyst writing a short performance summary for a non-technical
stakeholder. Use ONLY the numbers in the table below -- do not invent figures.

Write 3-5 sentences highlighting the most notable differences between the campaigns
(e.g. open rate, click rate), in plain English with no jargon.

Table (CSV):
{table_csv}

Narrative:
""")

campaign_narrative_llm = get_llm("campaign_narrative")
campaign_narrative_chain = campaign_narrative_prompt | campaign_narrative_llm | StrOutputParser()

def generate_campaign_performance_narrative(question):
    """Reuses Task 2's run_text_to_sql end-to-end, then narrates the result."""
    sql, result = run_text_to_sql(question)

    if not isinstance(result, pd.DataFrame):
        return {"sql": sql, "result": result, "narrative": f"Could not generate a narrative: {result}", "key_metrics": {}}

    key_metrics = result.set_index(result.columns[0]).to_dict(orient="index")
    raw_narrative = campaign_narrative_chain.invoke({"table_csv": result.to_csv(index=False)})

    return {"sql": sql, "result": result, "narrative": raw_narrative.strip(), "key_metrics": key_metrics}

print("✅ Campaign performance narrative chain ready")

In [ ]:
question7 = (
    "For each of 'Tet2026 Email Blast' and 'Tet2025 Email Blast', show total sends, "
    "the open rate (opened divided by total sends), and the click rate "
    "(clicked divided by total sends)."
)
narrative_result = generate_campaign_performance_narrative(question7)

show("TASK 7 · Campaign Performance Narrative", question7, f"SQL: {narrative_result['sql']}")
if isinstance(narrative_result["result"], pd.DataFrame):
    display(narrative_result["result"])
print(f"\n📝 Narrative:\n{narrative_result['narrative']}")
print(f"\n📊 key_metrics: {json.dumps(narrative_result['key_metrics'], indent=2, ensure_ascii=False)}")

## Task 8 — Schema-Mapping Assistant: Client Onboarding

**Description:** RAG over a client's raw source columns and CDP schema documentation to propose field mappings, while flagging ambiguous mappings for human review instead of silently guessing.

**Flow:**
1. For each raw column name, retrieve the top candidate canonical schema fields via `schema_retriever` (reuses the same semantic index Task 2 built from `schema_docs` — no separate corpus needed).
2. Prompt the model with the raw column name + candidate canonical fields → propose the best mapping, a confidence rating, and a short note.
3. Aggregate all column mappings into a single review table.

**Input metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `raw_columns` | `list[str]` | `["Cust_ID", "E-mail Addr", "Reward Points Balance"]` | Column headers from the new client's file, in whatever naming convention they used |

**Output metadata**

| Field | Type | Example | Notes |
|---|---|---|---|
| `raw_column` | `str` | `"E-mail Addr"` | Echoed input |
| `mapped_table` | `str` | `"customers"` | Best-guess canonical table, or `null` if no good match |
| `mapped_column` | `str` | `"email"` | Best-guess canonical column |
| `confidence` | `str` | `"High"` | `High` / `Medium` / `Low` |
| `notes` | `str` | `"Direct match"` | Explains ambiguity or why confidence is low — this is what a human reviewer reads first, so nothing gets silently guessed |

In [ ]:
# ============================================================
# TASK 8 — Schema-Mapping Assistant: Client Onboarding
#
# Reuses `schema_retriever` and `schema_docs` from Task 2 -- the same
# canonical-field index, no separate corpus needed.
# ============================================================

schema_mapping_prompt = ChatPromptTemplate.from_template("""
You are a data-onboarding assistant mapping a new client's raw column name onto our
canonical CDP schema. Use ONLY the candidate canonical fields listed in the Context below.

Return ONLY a JSON object with exactly these keys:
- "mapped_table": the canonical table name, or null if nothing in the Context is a good match
- "mapped_column": the canonical column name, or null
- "confidence": "High", "Medium", or "Low"
- "notes": one short sentence -- if confidence is Medium or Low, explain the ambiguity so a
  human reviewer knows what to check

Candidate canonical fields (from schema search):
{context}

Raw column name to map: {question}

JSON:
""")

schema_mapping_llm = get_llm("schema_mapping")
schema_mapping_chain = make_rag_chain(schema_retriever, schema_mapping_llm, schema_mapping_prompt)

def map_schema_column(raw_column_name):
    raw = schema_mapping_chain.invoke(raw_column_name)
    result = parse_json_safely(raw)
    result["raw_column"] = raw_column_name
    return result

print("✅ Schema-mapping assistant ready")

In [ ]:
# A new client's raw CSV headers -- deliberately messy naming, and "Loyalty Tier"
# has no canonical equivalent, to test the low-confidence / flag-for-review path.
RAW_COLUMNS = [
    "Cust_ID",
    "Full Name",
    "E-mail Addr",
    "Reward Points Balance",
    "Sign-up Dt",
    "Home City",
    "Seg",
    "Camp_ID",
    "Send_TS",
    "Msg_Opened",
    "Loyalty Tier",
]

mapping_results = [map_schema_column(c) for c in RAW_COLUMNS]
mapping_df = pd.DataFrame(mapping_results)
cols = ["raw_column", "mapped_table", "mapped_column", "confidence", "notes"]
mapping_df = mapping_df[[c for c in cols if c in mapping_df.columns]]

print(f"\n{'=' * 80}\nTASK 8 · Schema-Mapping Assistant — {len(mapping_df)} raw columns mapped\n{'=' * 80}")
display(mapping_df)


## Task 9 — Persona Decision Loop Demo: Experience → Model → Lens → Perception → Interpretation → Action → New Experience


For **Use Case 9**, the conceptual flow is:

```text
Experience
   ↓
Learned Model
   ↓
Kernel / Lens
   ↓
Persona
   ↓
Perception
   ↓
Interpretation
   ↓
Action
   ↓
New Experience
   ↺
```

For the iPhone 18 demo specifically:

```text
Ad Impression
      +
Shopping Mall Visit
      +
Product Demo
      +
iPhone 15 vs iPhone 18 Comparison
            ↓
      Learned Signals
            ↓
   Persona / Value Lens
            ↓
 "I don't see enough difference"
            ↓
      Interpretation
            ↓
 "Do not push upgrade yet"
            ↓
      Keep using iPhone 15
            ↓
    New Customer Experience
            ↺
```

This makes **Use Case 9** materially different from Campaign Performance Narrative: #7 explains **what happened in aggregate**, while #9 reasons about **what is happening with an individual customer and what should happen next**.


### What the demo shows

- **Experience** = raw customer evidence from Customer 360, events, campaigns and support.
- **Learned Model** = measurable patterns derived from the evidence (activity, purchase value, recency, engagement, support history).
- **Kernel / Lens** = a dynamic Persona hypothesis: the lens we use to understand the customer.
- **Perception** = what LEO believes is happening **right now**, given the latest context.
- **Interpretation** = what the current state may mean for a business decision.
- **Action** = a contextual next-best action for the customer.
- **New Experience** = a simulated outcome that would become new evidence and update the Persona again.

> Important: only the first two stages are directly observed/calculated from the sample database. Persona, perception and interpretation are **AI-generated hypotheses grounded in those signals**, not facts about a real person.


In [ ]:
# ============================================================
# TASK 9 — Persona Decision Loop Demo
# ============================================================
# Refactored into an OOP-style demo pipeline:
#
#   1. Seed realistic marketing scenarios into SQLite
#   2. Load ALL evidence from SQLite into Pandas
#   3. Compute deterministic metrics / learned signals
#   4. Build a compact, rich customer context
#   5. Send ONLY the rich context to the LLM
#   6. Generate Persona → Perception → Interpretation → Action
#   7. Render Colab-friendly tables
#
# Product:
#   iPhone 18
#
# Current device:
#   iPhone 15
#
# Initial analysis date:
#   2026-09-27
#
# IMPORTANT:
#   Future outcomes are stored in SQLite but are NOT included in the
#   initial LLM context. This prevents data leakage.
#
# Existing notebook dependencies expected:
#   get_llm(...)
#   parse_json_safely(...)
# ============================================================

import json
import sqlite3
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

import pandas as pd
from IPython.display import Markdown, display

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 0. Configuration
# ============================================================

class DemoConfig:
    AS_OF = "2026-09-27"
    PRODUCT = "iPhone 18"
    CURRENT_PRODUCT = "iPhone 15"

    # Six-month future outcome used by Profile 3.
    FUTURE_OUTCOME_DATE = "2027-03-27"

    # Colab-friendly SQLite file.
    DB_PATH = "/content/leo_persona_demo.sqlite"

    # Existing LEO model alias from the notebook.
    LLM_ALIAS = "analytics_persona"

    LLM_TEMPERATURE = 0.2
    LLM_MAX_TOKENS = 2000


# ============================================================
# 1. Seed data models
# ============================================================

@dataclass
class EventSpec:
    event_id: str
    customer_id: int
    event_name: str
    event_time: str
    channel: str
    value: float = 0
    product: Optional[str] = None
    details: str = ""

@dataclass
class CampaignSendSpec:
    send_id: str
    customer_id: int
    campaign_id: int
    sent_at: str
    opened: int
    clicked: int

@dataclass
class TicketSpec:
    ticket_id: str
    customer_id: int
    channel: str
    subject: str
    message: str
    created_at: str

@dataclass
class FutureOutcomeSpec:
    customer_id: int
    outcome_date: str
    event_name: str
    channel: str
    product: str
    value: float
    status_after: str
    details: str

@dataclass
class CustomerSpec:
    customer_id: int
    full_name: str
    city: str
    signup_date: Optional[str]
    ltv: float
    status: str
    profile_type: str
    segments: List[str] = field(default_factory=list)
    events: List[EventSpec] = field(default_factory=list)
    campaigns: List[CampaignSendSpec] = field(default_factory=list)
    tickets: List[TicketSpec] = field(default_factory=list)
    future_outcome: Optional[FutureOutcomeSpec] = None


# ============================================================
# 2. Scenario factory
# ============================================================

class DemoScenarioFactory:
    """
    Creates realistic synthetic marketing scenarios.

    13 profiles:
      1-3  = original demo profiles
      4-13 = ten additional common marketing scenarios
    """

    def __init__(self, config: DemoConfig):
        self.config = config

    def _event(
        self,
        customer_id: int,
        suffix: int,
        event_name: str,
        event_time: str,
        channel: str,
        value: float = 0,
        product: Optional[str] = None,
        details: str = "",
    ) -> EventSpec:
        return EventSpec(
            event_id=f"e{customer_id}-{suffix:02d}",
            customer_id=customer_id,
            event_name=event_name,
            event_time=event_time,
            channel=channel,
            value=value,
            product=product,
            details=details,
        )

    def _campaign(
        self,
        customer_id: int,
        send_suffix: int,
        campaign_id: int,
        sent_at: str,
        opened: int,
        clicked: int,
    ) -> CampaignSendSpec:
        return CampaignSendSpec(
            send_id=f"send-{customer_id}-{send_suffix}",
            customer_id=customer_id,
            campaign_id=campaign_id,
            sent_at=sent_at,
            opened=opened,
            clicked=clicked,
        )

    def build(self) -> List[CustomerSpec]:
        return [
            self._profile_1_existing_customer(),
            self._profile_2_lead(),
            self._profile_3_future_customer(),

            self._profile_4_iphone18_converter(),
            self._profile_5_price_sensitive(),
            self._profile_6_abandoned_cart(),
            self._profile_7_churn_risk(),
            self._profile_8_cross_sell(),
            self._profile_9_ad_fatigue(),
            self._profile_10_competitor_compare(),
            self._profile_11_high_intent_upgrade(),
            self._profile_12_seasonal_gift(),
            self._profile_13_advocate_referral(),
        ]

    # --------------------------------------------------------
    # 1. Existing customer — interested but keeps iPhone 15
    # --------------------------------------------------------

    def _profile_1_existing_customer(self) -> CustomerSpec:
        cid = 1001

        return CustomerSpec(
            customer_id=cid,
            full_name="Alex Nguyen",
            city="Ho Chi Minh City",
            signup_date="2026-08-03",
            ltv=42_000_000,
            status="customer",
            profile_type="converted_customer",
            segments=[
                "Apple Ecosystem",
                "High Engagement",
                "Existing Customer",
            ],
            events=[
                self._event(
                    cid, 1, "mall_visit", "2026-09-18 18:20:00",
                    "Shopping Mall",
                    details="Visited electronics area after work",
                ),
                self._event(
                    cid, 2, "product_demo", "2026-09-18 18:35:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Viewed and tried iPhone 18 in store",
                ),
                self._event(
                    cid, 3, "product_compare", "2026-09-18 18:42:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared iPhone 18 with current iPhone 15",
                ),
                self._event(
                    cid, 4, "ad_impression", "2026-09-20 09:15:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 launch advertisement",
                ),
                self._event(
                    cid, 5, "ad_click", "2026-09-20 09:18:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Clicked iPhone 18 advertisement",
                ),
                self._event(
                    cid, 6, "purchase", "2026-09-21 15:30:00",
                    "Retail Store",
                    value=8_900_000,
                    product="Apple Watch",
                    details="Purchased Apple Watch, did not upgrade phone",
                ),
                self._event(
                    cid, 7, "support_contact", "2026-09-23 10:10:00",
                    "Customer Service",
                    product=self.config.CURRENT_PRODUCT,
                    details="Asked about continuing to use iPhone 15",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 1, "2026-09-20 09:15:00", 1, 1),
                self._campaign(cid, 2, 2, "2026-09-21 08:00:00", 1, 1),
            ],
            tickets=[
                TicketSpec(
                    ticket_id="t1001-01",
                    customer_id=cid,
                    channel="Customer Service",
                    subject="Continue using iPhone 15",
                    message=(
                        "Customer asked whether there is a meaningful "
                        "reason to replace the current iPhone 15."
                    ),
                    created_at="2026-09-23 10:10:00",
                )
            ],
        )

    # --------------------------------------------------------
    # 2. Lead — ad + mall interest, no conversion
    # --------------------------------------------------------

    def _profile_2_lead(self) -> CustomerSpec:
        cid = 1002

        return CustomerSpec(
            customer_id=cid,
            full_name="Mai Tran",
            city="Ho Chi Minh City",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="lead",
            segments=[
                "iPhone Interest",
                "Price Sensitive",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_impression", "2026-09-19 20:10:00",
                    "TikTok Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 advertisement",
                ),
                self._event(
                    cid, 2, "ad_click", "2026-09-19 20:13:00",
                    "TikTok Ads",
                    product=self.config.PRODUCT,
                    details="Clicked iPhone 18 advertisement",
                ),
                self._event(
                    cid, 3, "mall_visit", "2026-09-22 14:25:00",
                    "Shopping Mall",
                    details="Visited electronics store",
                ),
                self._event(
                    cid, 4, "product_demo", "2026-09-22 14:35:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Held and tested iPhone 18",
                ),
                self._event(
                    cid, 5, "product_compare", "2026-09-22 14:42:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared iPhone 18 with iPhone 15",
                ),
                self._event(
                    cid, 6, "page_view", "2026-09-24 21:05:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed iPhone 18 specification page",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 1, "2026-09-19 20:10:00", 1, 1),
            ],
        )

    # --------------------------------------------------------
    # 3. Lead now → iPhone 18 customer six months later
    # --------------------------------------------------------

    def _profile_3_future_customer(self) -> CustomerSpec:
        cid = 1003

        return CustomerSpec(
            customer_id=cid,
            full_name="David Le",
            city="Ho Chi Minh City",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="future_customer",
            segments=[
                "iPhone Interest",
                "Research Heavy",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_impression", "2026-09-17 19:10:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 advertisement",
                ),
                self._event(
                    cid, 2, "ad_click", "2026-09-17 19:14:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Clicked iPhone 18 advertisement",
                ),
                self._event(
                    cid, 3, "mall_visit", "2026-09-20 16:20:00",
                    "Shopping Mall",
                    details="Visited premium electronics store",
                ),
                self._event(
                    cid, 4, "product_demo", "2026-09-20 16:32:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Tried iPhone 18 in store",
                ),
                self._event(
                    cid, 5, "product_compare", "2026-09-20 16:40:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared iPhone 18 with current iPhone 15",
                ),
                self._event(
                    cid, 6, "page_view", "2026-09-25 21:10:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Read iPhone 18 product information",
                ),
                self._event(
                    cid, 7, "comparison_content", "2026-09-26 22:15:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed iPhone 15 vs iPhone 18 comparison",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 1, "2026-09-17 19:10:00", 1, 1),
                self._campaign(cid, 2, 3, "2026-09-25 08:00:00", 1, 1),
            ],
            future_outcome=FutureOutcomeSpec(
                customer_id=cid,
                outcome_date=self.config.FUTURE_OUTCOME_DATE,
                event_name="purchase",
                channel="Retail Store",
                product=self.config.PRODUCT,
                value=32_990_000,
                status_after="customer",
                details=(
                    "Six months later, customer upgraded from "
                    "iPhone 15 to iPhone 18."
                ),
            ),
        )

    # --------------------------------------------------------
    # 4. Actual iPhone 18 conversion
    # --------------------------------------------------------

    def _profile_4_iphone18_converter(self) -> CustomerSpec:
        cid = 1004

        return CustomerSpec(
            customer_id=cid,
            full_name="Linh Pham",
            city="Hanoi",
            signup_date="2026-07-12",
            ltv=68_000_000,
            status="customer",
            profile_type="iphone18_converter",
            segments=[
                "Early Adopter",
                "High Intent",
                "Apple Ecosystem",
            ],
            events=[
                self._event(
                    cid, 1, "ad_impression", "2026-09-15 08:10:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 launch ad",
                ),
                self._event(
                    cid, 2, "ad_click", "2026-09-15 08:12:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Clicked launch ad",
                ),
                self._event(
                    cid, 3, "product_demo", "2026-09-17 17:20:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Requested an in-store demo",
                ),
                self._event(
                    cid, 4, "product_compare", "2026-09-17 17:31:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared current phone with iPhone 18",
                ),
                self._event(
                    cid, 5, "checkout", "2026-09-18 11:14:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Started online checkout",
                ),
                self._event(
                    cid, 6, "purchase", "2026-09-18 11:24:00",
                    "Website",
                    value=36_990_000,
                    product=self.config.PRODUCT,
                    details="Purchased iPhone 18",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 1, "2026-09-15 08:10:00", 1, 1),
                self._campaign(cid, 2, 4, "2026-09-17 08:00:00", 1, 1),
            ],
        )

    # --------------------------------------------------------
    # 5. Price-sensitive prospect
    # --------------------------------------------------------

    def _profile_5_price_sensitive(self) -> CustomerSpec:
        cid = 1005

        return CustomerSpec(
            customer_id=cid,
            full_name="Huy Vo",
            city="Da Nang",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="price_sensitive",
            segments=[
                "Price Sensitive",
                "Promotion Seeker",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_impression", "2026-09-16 10:05:00",
                    "Google Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 price ad",
                ),
                self._event(
                    cid, 2, "page_view", "2026-09-16 10:07:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed product price page",
                ),
                self._event(
                    cid, 3, "price_check", "2026-09-18 19:40:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Checked price and financing options",
                ),
                self._event(
                    cid, 4, "promotion_view", "2026-09-21 20:05:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed discount campaign",
                ),
                self._event(
                    cid, 5, "page_view", "2026-09-25 20:50:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Returned to price page",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 5, "2026-09-16 10:05:00", 1, 1),
                self._campaign(cid, 2, 6, "2026-09-21 20:05:00", 1, 0),
            ],
        )

    # --------------------------------------------------------
    # 6. High intent but abandoned cart
    # --------------------------------------------------------

    def _profile_6_abandoned_cart(self) -> CustomerSpec:
        cid = 1006

        return CustomerSpec(
            customer_id=cid,
            full_name="Thao Nguyen",
            city="Ho Chi Minh City",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="abandoned_cart",
            segments=[
                "High Intent",
                "Cart Abandoner",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_click", "2026-09-19 08:20:00",
                    "Instagram Ads",
                    product=self.config.PRODUCT,
                    details="Clicked product ad",
                ),
                self._event(
                    cid, 2, "product_view", "2026-09-19 08:24:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed product page",
                ),
                self._event(
                    cid, 3, "product_compare", "2026-09-19 08:34:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Compared iPhone 18 with iPhone 15",
                ),
                self._event(
                    cid, 4, "add_to_cart", "2026-09-19 08:41:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Added iPhone 18 to cart",
                ),
                self._event(
                    cid, 5, "checkout", "2026-09-19 08:48:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Started checkout",
                ),
                self._event(
                    cid, 6, "cart_abandon", "2026-09-19 09:02:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Left checkout without purchase",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 7, "2026-09-19 08:20:00", 1, 1),
                self._campaign(cid, 2, 8, "2026-09-22 09:00:00", 1, 0),
            ],
        )

    # --------------------------------------------------------
    # 7. Existing customer with churn-risk signals
    # --------------------------------------------------------

    def _profile_7_churn_risk(self) -> CustomerSpec:
        cid = 1007

        return CustomerSpec(
            customer_id=cid,
            full_name="Quang Tran",
            city="Hai Phong",
            signup_date="2024-04-11",
            ltv=25_000_000,
            status="customer",
            profile_type="churn_risk",
            segments=[
                "Churn Risk",
                "Existing Customer",
                "Low Recent Engagement",
            ],
            events=[
                self._event(
                    cid, 1, "purchase", "2025-09-12 12:20:00",
                    "Retail Store",
                    value=14_000_000,
                    product=self.config.CURRENT_PRODUCT,
                    details="Purchased current iPhone 15",
                ),
                self._event(
                    cid, 2, "support_contact", "2026-08-20 10:10:00",
                    "Customer Service",
                    product=self.config.CURRENT_PRODUCT,
                    details="Reported battery concern",
                ),
                self._event(
                    cid, 3, "ad_impression", "2026-09-18 13:10:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 ad",
                ),
                self._event(
                    cid, 4, "page_view", "2026-09-18 13:12:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed iPhone 18 page",
                ),
                self._event(
                    cid, 5, "support_contact", "2026-09-24 09:30:00",
                    "Customer Service",
                    product=self.config.CURRENT_PRODUCT,
                    details="Asked about repair and battery service",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 9, "2026-09-18 13:10:00", 1, 0),
            ],
            tickets=[
                TicketSpec(
                    ticket_id="t1007-01",
                    customer_id=cid,
                    channel="Customer Service",
                    subject="Battery service for iPhone 15",
                    message="Customer asked about battery service options.",
                    created_at="2026-09-24 09:30:00",
                )
            ],
        )

    # --------------------------------------------------------
    # 8. Cross-sell / ecosystem expansion
    # --------------------------------------------------------

    def _profile_8_cross_sell(self) -> CustomerSpec:
        cid = 1008

        return CustomerSpec(
            customer_id=cid,
            full_name="Nhi Le",
            city="Can Tho",
            signup_date="2025-06-08",
            ltv=31_500_000,
            status="customer",
            profile_type="cross_sell",
            segments=[
                "Apple Ecosystem",
                "Cross-sell Opportunity",
                "Engaged Customer",
            ],
            events=[
                self._event(
                    cid, 1, "purchase", "2026-07-18 15:20:00",
                    "Retail Store",
                    value=8_500_000,
                    product="AirPods Pro",
                    details="Purchased AirPods Pro",
                ),
                self._event(
                    cid, 2, "purchase", "2026-08-14 13:05:00",
                    "Website",
                    value=8_900_000,
                    product="Apple Watch",
                    details="Purchased Apple Watch",
                ),
                self._event(
                    cid, 3, "ad_impression", "2026-09-16 17:05:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 ecosystem campaign",
                ),
                self._event(
                    cid, 4, "page_view", "2026-09-16 17:12:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed ecosystem bundle",
                ),
                self._event(
                    cid, 5, "recommendation_click", "2026-09-21 18:15:00",
                    "App",
                    product="Apple Watch",
                    details="Clicked accessory recommendation",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 10, "2026-09-16 17:05:00", 1, 1),
            ],
        )

    # --------------------------------------------------------
    # 9. Ad fatigue
    # --------------------------------------------------------

    def _profile_9_ad_fatigue(self) -> CustomerSpec:
        cid = 1009

        events = [
            self._event(
                cid, 1, "ad_impression", "2026-09-17 08:00:00",
                "Meta Ads",
                product=self.config.PRODUCT,
                details="Repeated launch ad impression",
            ),
            self._event(
                cid, 2, "ad_impression", "2026-09-18 08:00:00",
                "Meta Ads",
                product=self.config.PRODUCT,
                details="Repeated launch ad impression",
            ),
            self._event(
                cid, 3, "ad_impression", "2026-09-19 08:00:00",
                "Meta Ads",
                product=self.config.PRODUCT,
                details="Repeated launch ad impression",
            ),
            self._event(
                cid, 4, "ad_impression", "2026-09-20 08:00:00",
                "Meta Ads",
                product=self.config.PRODUCT,
                details="Repeated launch ad impression",
            ),
            self._event(
                cid, 5, "ad_impression", "2026-09-21 08:00:00",
                "Meta Ads",
                product=self.config.PRODUCT,
                details="Repeated launch ad impression",
            ),
            self._event(
                cid, 6, "page_view", "2026-09-23 18:00:00",
                "Website",
                product=self.config.PRODUCT,
                details="One brief product page visit",
            ),
        ]

        return CustomerSpec(
            customer_id=cid,
            full_name="Bao Pham",
            city="Ho Chi Minh City",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="ad_fatigue",
            segments=[
                "Ad Fatigue",
                "Low Engagement",
                "Prospect",
            ],
            events=events,
            campaigns=[
                self._campaign(cid, 1, 11, "2026-09-17 08:00:00", 0, 0),
                self._campaign(cid, 2, 11, "2026-09-18 08:00:00", 0, 0),
                self._campaign(cid, 3, 11, "2026-09-19 08:00:00", 0, 0),
                self._campaign(cid, 4, 11, "2026-09-20 08:00:00", 0, 0),
                self._campaign(cid, 5, 11, "2026-09-21 08:00:00", 0, 0),
            ],
        )

    # --------------------------------------------------------
    # 10. Competitor comparison
    # --------------------------------------------------------

    def _profile_10_competitor_compare(self) -> CustomerSpec:
        cid = 1010

        return CustomerSpec(
            customer_id=cid,
            full_name="Minh Ho",
            city="Hanoi",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="competitor_compare",
            segments=[
                "Competitive Shopper",
                "Research Heavy",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_click", "2026-09-18 14:05:00",
                    "Google Ads",
                    product=self.config.PRODUCT,
                    details="Clicked iPhone 18 ad",
                ),
                self._event(
                    cid, 2, "comparison_content", "2026-09-18 14:18:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed iPhone 18 vs competing Android page",
                ),
                self._event(
                    cid, 3, "page_view", "2026-09-20 10:10:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Read product reviews",
                ),
                self._event(
                    cid, 4, "mall_visit", "2026-09-21 15:35:00",
                    "Shopping Mall",
                    details="Visited several smartphone stores",
                ),
                self._event(
                    cid, 5, "product_demo", "2026-09-21 15:50:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Tried iPhone 18",
                ),
                self._event(
                    cid, 6, "product_compare", "2026-09-21 16:05:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared iPhone 18 against other brands",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 12, "2026-09-18 14:05:00", 1, 1),
            ],
        )

    # --------------------------------------------------------
    # 11. Existing customer — strong upgrade need
    # --------------------------------------------------------

    def _profile_11_high_intent_upgrade(self) -> CustomerSpec:
        cid = 1011

        return CustomerSpec(
            customer_id=cid,
            full_name="Trang Bui",
            city="Ho Chi Minh City",
            signup_date="2025-01-14",
            ltv=39_000_000,
            status="customer",
            profile_type="high_intent_upgrade",
            segments=[
                "High Intent",
                "Existing Customer",
                "Upgrade Candidate",
            ],
            events=[
                self._event(
                    cid, 1, "purchase", "2025-02-18 12:00:00",
                    "Retail Store",
                    value=29_000_000,
                    product=self.config.CURRENT_PRODUCT,
                    details="Purchased iPhone 15",
                ),
                self._event(
                    cid, 2, "support_contact", "2026-09-10 10:30:00",
                    "Customer Service",
                    product=self.config.CURRENT_PRODUCT,
                    details="Reported current device issue",
                ),
                self._event(
                    cid, 3, "mall_visit", "2026-09-17 18:10:00",
                    "Shopping Mall",
                    details="Visited smartphone store",
                ),
                self._event(
                    cid, 4, "product_demo", "2026-09-17 18:25:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Requested iPhone 18 demo",
                ),
                self._event(
                    cid, 5, "product_compare", "2026-09-17 18:31:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Compared against current iPhone 15",
                ),
                self._event(
                    cid, 6, "checkout", "2026-09-20 20:20:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Started checkout after researching upgrade",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 13, "2026-09-17 18:10:00", 1, 1),
            ],
            tickets=[
                TicketSpec(
                    ticket_id="t1011-01",
                    customer_id=cid,
                    channel="Customer Service",
                    subject="Current iPhone issue",
                    message="Customer reported an issue with current device.",
                    created_at="2026-09-10 10:30:00",
                )
            ],
        )

    # --------------------------------------------------------
    # 12. Seasonal / gift purchase intent
    # --------------------------------------------------------

    def _profile_12_seasonal_gift(self) -> CustomerSpec:
        cid = 1012

        return CustomerSpec(
            customer_id=cid,
            full_name="Khoa Nguyen",
            city="Nha Trang",
            signup_date=None,
            ltv=0,
            status="lead",
            profile_type="seasonal_gift",
            segments=[
                "Gift Shopper",
                "Seasonal Intent",
                "Prospect",
            ],
            events=[
                self._event(
                    cid, 1, "ad_impression", "2026-09-08 12:10:00",
                    "Facebook Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 gift campaign",
                ),
                self._event(
                    cid, 2, "page_view", "2026-09-08 12:15:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed gift campaign page",
                ),
                self._event(
                    cid, 3, "page_view", "2026-09-10 20:30:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Returned to product page",
                ),
                self._event(
                    cid, 4, "store_locator", "2026-09-22 09:10:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Checked nearby store",
                ),
                self._event(
                    cid, 5, "mall_visit", "2026-09-24 17:10:00",
                    "Shopping Mall",
                    details="Visited retail store",
                ),
                self._event(
                    cid, 6, "product_demo", "2026-09-24 17:18:00",
                    "Shopping Mall",
                    product=self.config.PRODUCT,
                    details="Asked staff about the product",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 14, "2026-09-08 12:10:00", 1, 1),
                self._campaign(cid, 2, 15, "2026-09-22 09:10:00", 1, 0),
            ],
        )

    # --------------------------------------------------------
    # 13. Existing advocate / referral
    # --------------------------------------------------------

    def _profile_13_advocate_referral(self) -> CustomerSpec:
        cid = 1013

        return CustomerSpec(
            customer_id=cid,
            full_name="Yen Do",
            city="Ho Chi Minh City",
            signup_date="2024-09-05",
            ltv=75_000_000,
            status="customer",
            profile_type="advocate_referral",
            segments=[
                "Brand Advocate",
                "High LTV",
                "Referral Opportunity",
            ],
            events=[
                self._event(
                    cid, 1, "purchase", "2026-01-18 14:20:00",
                    "Retail Store",
                    value=29_000_000,
                    product=self.config.CURRENT_PRODUCT,
                    details="Purchased iPhone 15",
                ),
                self._event(
                    cid, 2, "ad_impression", "2026-09-15 11:00:00",
                    "Meta Ads",
                    product=self.config.PRODUCT,
                    details="Saw iPhone 18 campaign",
                ),
                self._event(
                    cid, 3, "social_share", "2026-09-16 09:30:00",
                    "Instagram",
                    product=self.config.PRODUCT,
                    details="Shared product campaign content",
                ),
                self._event(
                    cid, 4, "referral", "2026-09-17 12:10:00",
                    "Referral",
                    product=self.config.PRODUCT,
                    details="Referred a friend to store",
                ),
                self._event(
                    cid, 5, "page_view", "2026-09-20 14:40:00",
                    "Website",
                    product=self.config.PRODUCT,
                    details="Viewed iPhone 18 page",
                ),
            ],
            campaigns=[
                self._campaign(cid, 1, 16, "2026-09-15 11:00:00", 1, 1),
            ],
        )


# ============================================================
# 3. SQLite database
# ============================================================

class DemoDatabase:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.connection: Optional[sqlite3.Connection] = None

    def connect(self):
        self.connection = sqlite3.connect(
            self.db_path
        )
        self.connection.execute(
            "PRAGMA foreign_keys = ON"
        )
        return self.connection

    def close(self):
        if self.connection is not None:
            self.connection.close()
            self.connection = None

    def reset(self):
        if self.connection is None:
            self.connect()

        tables = [
            "segment_members",
            "segments",
            "campaign_sends",
            "campaigns",
            "support_tickets",
            "future_outcomes",
            "events",
            "customers",
        ]

        for table in tables:
            self.connection.execute(
                f"DROP TABLE IF EXISTS {table}"
            )

        self.connection.commit()

    def create_schema(self):
        if self.connection is None:
            self.connect()

        self.connection.executescript(
            """
            CREATE TABLE customers (
                customer_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                city TEXT NOT NULL,
                signup_date TEXT,
                ltv REAL DEFAULT 0,
                status TEXT NOT NULL,
                profile_type TEXT NOT NULL,
                current_product TEXT
            );

            CREATE TABLE events (
                event_id TEXT PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                event_name TEXT NOT NULL,
                event_time TEXT NOT NULL,
                channel TEXT NOT NULL,
                value REAL DEFAULT 0,
                product TEXT,
                details TEXT,
                FOREIGN KEY (customer_id)
                    REFERENCES customers(customer_id)
            );

            CREATE TABLE campaigns (
                campaign_id INTEGER PRIMARY KEY,
                campaign_name TEXT NOT NULL,
                channel TEXT NOT NULL,
                start_date TEXT NOT NULL
            );

            CREATE TABLE campaign_sends (
                send_id TEXT PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                campaign_id INTEGER NOT NULL,
                sent_at TEXT NOT NULL,
                opened INTEGER DEFAULT 0,
                clicked INTEGER DEFAULT 0,
                FOREIGN KEY (customer_id)
                    REFERENCES customers(customer_id),
                FOREIGN KEY (campaign_id)
                    REFERENCES campaigns(campaign_id)
            );

            CREATE TABLE support_tickets (
                ticket_id TEXT PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                channel TEXT NOT NULL,
                subject TEXT NOT NULL,
                message TEXT,
                created_at TEXT NOT NULL,
                FOREIGN KEY (customer_id)
                    REFERENCES customers(customer_id)
            );

            CREATE TABLE segments (
                segment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                segment_name TEXT NOT NULL UNIQUE
            );

            CREATE TABLE segment_members (
                customer_id INTEGER NOT NULL,
                segment_id INTEGER NOT NULL,
                PRIMARY KEY (customer_id, segment_id),
                FOREIGN KEY (customer_id)
                    REFERENCES customers(customer_id),
                FOREIGN KEY (segment_id)
                    REFERENCES segments(segment_id)
            );

            CREATE TABLE future_outcomes (
                customer_id INTEGER PRIMARY KEY,
                outcome_date TEXT NOT NULL,
                event_name TEXT NOT NULL,
                channel TEXT NOT NULL,
                product TEXT,
                value REAL DEFAULT 0,
                status_after TEXT,
                details TEXT,
                FOREIGN KEY (customer_id)
                    REFERENCES customers(customer_id)
            );
            """
        )

        self.connection.commit()

    def seed(
        self,
        customers: List[CustomerSpec],
    ):
        if self.connection is None:
            self.connect()

        campaign_catalog = {
            1: (
                "iPhone 18 Launch",
                "Meta Ads",
                "2026-09-15",
            ),
            2: (
                "Apple Ecosystem Upgrade",
                "Email",
                "2026-09-21",
            ),
            3: (
                "iPhone 18 Comparison",
                "Email",
                "2026-09-25",
            ),
            4: (
                "Early Access",
                "Email",
                "2026-09-17",
            ),
            5: (
                "iPhone 18 Price Watch",
                "Google Ads",
                "2026-09-16",
            ),
            6: (
                "Promotion Reminder",
                "Email",
                "2026-09-21",
            ),
            7: (
                "iPhone 18 Cart Reminder",
                "Instagram Ads",
                "2026-09-19",
            ),
            8: (
                "Cart Recovery",
                "Email",
                "2026-09-22",
            ),
            9: (
                "iPhone 18 Awareness",
                "Meta Ads",
                "2026-09-18",
            ),
            10: (
                "Apple Ecosystem",
                "Meta Ads",
                "2026-09-16",
            ),
            11: (
                "iPhone 18 Launch Frequency",
                "Meta Ads",
                "2026-09-17",
            ),
            12: (
                "Comparison Campaign",
                "Google Ads",
                "2026-09-18",
            ),
            13: (
                "Upgrade Journey",
                "Meta Ads",
                "2026-09-17",
            ),
            14: (
                "Gift Campaign",
                "Facebook Ads",
                "2026-09-08",
            ),
            15: (
                "Store Locator",
                "Email",
                "2026-09-22",
            ),
            16: (
                "Advocate Campaign",
                "Meta Ads",
                "2026-09-15",
            ),
        }

        for campaign_id, (name, channel, start_date) in campaign_catalog.items():

            self.connection.execute(
                """
                INSERT INTO campaigns (
                    campaign_id,
                    campaign_name,
                    channel,
                    start_date
                )
                VALUES (?, ?, ?, ?)
                """,
                (
                    campaign_id,
                    name,
                    channel,
                    start_date,
                ),
            )

        for customer in customers:

            self.connection.execute(
                """
                INSERT INTO customers (
                    customer_id,
                    full_name,
                    city,
                    signup_date,
                    ltv,
                    status,
                    profile_type,
                    current_product
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    customer.customer_id,
                    customer.full_name,
                    customer.city,
                    customer.signup_date,
                    customer.ltv,
                    customer.status,
                    customer.profile_type,
                    self.config_current_product(),
                ),
            )

            for event in customer.events:

                self.connection.execute(
                    """
                    INSERT INTO events (
                        event_id,
                        customer_id,
                        event_name,
                        event_time,
                        channel,
                        value,
                        product,
                        details
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        event.event_id,
                        event.customer_id,
                        event.event_name,
                        event.event_time,
                        event.channel,
                        event.value,
                        event.product,
                        event.details,
                    ),
                )

            for campaign_send in customer.campaigns:

                self.connection.execute(
                    """
                    INSERT INTO campaign_sends (
                        send_id,
                        customer_id,
                        campaign_id,
                        sent_at,
                        opened,
                        clicked
                    )
                    VALUES (?, ?, ?, ?, ?, ?)
                    """,
                    (
                        campaign_send.send_id,
                        campaign_send.customer_id,
                        campaign_send.campaign_id,
                        campaign_send.sent_at,
                        campaign_send.opened,
                        campaign_send.clicked,
                    ),
                )

            for ticket in customer.tickets:

                self.connection.execute(
                    """
                    INSERT INTO support_tickets (
                        ticket_id,
                        customer_id,
                        channel,
                        subject,
                        message,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?)
                    """,
                    (
                        ticket.ticket_id,
                        ticket.customer_id,
                        ticket.channel,
                        ticket.subject,
                        ticket.message,
                        ticket.created_at,
                    ),
                )

            if customer.future_outcome:

                outcome = customer.future_outcome

                self.connection.execute(
                    """
                    INSERT INTO future_outcomes (
                        customer_id,
                        outcome_date,
                        event_name,
                        channel,
                        product,
                        value,
                        status_after,
                        details
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        outcome.customer_id,
                        outcome.outcome_date,
                        outcome.event_name,
                        outcome.channel,
                        outcome.product,
                        outcome.value,
                        outcome.status_after,
                        outcome.details,
                    ),
                )

            for segment_name in customer.segments:

                self.connection.execute(
                    """
                    INSERT OR IGNORE INTO segments (
                        segment_name
                    )
                    VALUES (?)
                    """,
                    (segment_name,),
                )

                segment_id = self.connection.execute(
                    """
                    SELECT segment_id
                    FROM segments
                    WHERE segment_name = ?
                    """,
                    (segment_name,),
                ).fetchone()[0]

                self.connection.execute(
                    """
                    INSERT OR IGNORE INTO segment_members (
                        customer_id,
                        segment_id
                    )
                    VALUES (?, ?)
                    """,
                    (
                        customer.customer_id,
                        segment_id,
                    ),
                )

        self.connection.commit()

    @staticmethod
    def config_current_product() -> str:
        return DemoConfig.CURRENT_PRODUCT

    def table_counts(self) -> pd.DataFrame:

        tables = [
            "customers",
            "events",
            "campaigns",
            "campaign_sends",
            "support_tickets",
            "segments",
            "segment_members",
            "future_outcomes",
        ]

        rows = []

        for table in tables:

            count = self.connection.execute(
                f"SELECT COUNT(*) FROM {table}"
            ).fetchone()[0]

            rows.append(
                {
                    "Table": table,
                    "Rows": count,
                }
            )

        return pd.DataFrame(rows)


# ============================================================
# 4. Customer repository — SQLite → Pandas
# ============================================================

@dataclass
class CustomerExperience:
    profile: Dict[str, Any]
    events: pd.DataFrame
    campaigns: pd.DataFrame
    tickets: pd.DataFrame
    segments: List[str]


class CustomerRepository:
    """
    SQLite is the source of truth.
    Pandas is the analytics layer.
    """

    def __init__(
        self,
        connection: sqlite3.Connection,
        config: DemoConfig,
    ):
        self.connection = connection
        self.config = config

    def load_customer(
        self,
        customer_id: int,
        as_of: str,
    ) -> CustomerExperience:

        profile_df = pd.read_sql_query(
            """
            SELECT
                customer_id,
                full_name,
                city,
                signup_date,
                ltv,
                status,
                profile_type,
                current_product
            FROM customers
            WHERE customer_id = ?
            """,
            self.connection,
            params=[customer_id],
        )

        if profile_df.empty:
            raise ValueError(
                f"Customer {customer_id} not found"
            )

        events = pd.read_sql_query(
            """
            SELECT
                event_id,
                event_name,
                event_time,
                channel,
                value,
                product,
                details
            FROM events
            WHERE customer_id = ?
              AND event_time <= ?
            ORDER BY event_time
            """,
            self.connection,
            params=[
                customer_id,
                f"{as_of} 23:59:59",
            ],
        )

        campaigns = pd.read_sql_query(
            """
            SELECT
                cs.send_id,
                cs.sent_at,
                cs.opened,
                cs.clicked,
                c.campaign_name,
                c.channel
            FROM campaign_sends cs
            JOIN campaigns c
              ON c.campaign_id = cs.campaign_id
            WHERE cs.customer_id = ?
              AND cs.sent_at <= ?
            ORDER BY cs.sent_at
            """,
            self.connection,
            params=[
                customer_id,
                f"{as_of} 23:59:59",
            ],
        )

        tickets = pd.read_sql_query(
            """
            SELECT
                ticket_id,
                channel,
                subject,
                message,
                created_at
            FROM support_tickets
            WHERE customer_id = ?
              AND created_at <= ?
            ORDER BY created_at
            """,
            self.connection,
            params=[
                customer_id,
                f"{as_of} 23:59:59",
            ],
        )

        segments_df = pd.read_sql_query(
            """
            SELECT
                s.segment_name
            FROM segment_members sm
            JOIN segments s
              ON s.segment_id = sm.segment_id
            WHERE sm.customer_id = ?
            ORDER BY s.segment_name
            """,
            self.connection,
            params=[customer_id],
        )

        if not events.empty:
            events["event_time"] = pd.to_datetime(
                events["event_time"]
            )

        if not campaigns.empty:
            campaigns["sent_at"] = pd.to_datetime(
                campaigns["sent_at"]
            )

        if not tickets.empty:
            tickets["created_at"] = pd.to_datetime(
                tickets["created_at"]
            )

        return CustomerExperience(
            profile=profile_df.iloc[0].to_dict(),
            events=events,
            campaigns=campaigns,
            tickets=tickets,
            segments=segments_df[
                "segment_name"
            ].tolist(),
        )

    def load_future_outcome(
        self,
        customer_id: int,
    ) -> Optional[Dict[str, Any]]:

        df = pd.read_sql_query(
            """
            SELECT
                customer_id,
                outcome_date,
                event_name,
                channel,
                product,
                value,
                status_after,
                details
            FROM future_outcomes
            WHERE customer_id = ?
            """,
            self.connection,
            params=[customer_id],
        )

        if df.empty:
            return None

        return df.iloc[0].to_dict()


# ============================================================
# 5. Feature engineering
# ============================================================

class CustomerFeatureEngineer:
    """
    Converts raw Pandas data into deterministic learned signals.

    No LLM is used in this layer.
    """

    EVENT_WEIGHTS = {
        "ad_impression": 1,
        "ad_click": 2,
        "page_view": 1,
        "product_view": 1,
        "mall_visit": 2,
        "product_demo": 3,
        "product_compare": 3,
        "comparison_content": 2,
        "price_check": 2,
        "promotion_view": 2,
        "store_locator": 2,
        "recommendation_click": 2,
        "social_share": 3,
        "referral": 4,
        "add_to_cart": 4,
        "checkout": 5,
        "cart_abandon": 2,
        "purchase": 10,
        "support_contact": 2,
    }

    def __init__(self, config: DemoConfig):
        self.config = config

    def build(
        self,
        experience: CustomerExperience,
    ) -> Dict[str, Any]:

        events = experience.events.copy()
        campaigns = experience.campaigns.copy()
        tickets = experience.tickets.copy()

        if events.empty:
            events["event_time"] = pd.to_datetime([])
        else:
            events["event_time"] = pd.to_datetime(
                events["event_time"]
            )

        as_of = pd.Timestamp(
            self.config.AS_OF
        )

        last_event = (
            events["event_time"].max()
            if not events.empty
            else None
        )

        days_since_last_event = (
            (as_of - last_event.normalize()).days
            if last_event is not None
            else None
        )

        purchase_events = self._events_by_name(
            events,
            ["purchase"],
        )

        product_events = (
            events[
                events["product"].eq(
                    self.config.PRODUCT
                )
            ]
            if not events.empty
            else pd.DataFrame()
        )

        current_product_events = (
            events[
                events["product"].eq(
                    self.config.CURRENT_PRODUCT
                )
            ]
            if not events.empty
            else pd.DataFrame()
        )

        comparison_events = self._events_by_name(
            events,
            [
                "product_compare",
                "comparison_content",
            ],
        )

        ad_events = self._events_by_name(
            events,
            [
                "ad_impression",
                "ad_click",
            ],
        )

        mall_events = (
            events[
                events["channel"].eq(
                    "Shopping Mall"
                )
            ]
            if not events.empty
            else pd.DataFrame()
        )

        high_intent_events = self._events_by_name(
            events,
            [
                "add_to_cart",
                "checkout",
                "purchase",
            ],
        )

        support_events = self._events_by_name(
            events,
            ["support_contact"],
        )

        recent_30d = (
            events[
                events["event_time"]
                >= (
                    as_of
                    - pd.Timedelta(days=30)
                )
            ]
            if not events.empty
            else pd.DataFrame()
        )

        campaign_sent_count = int(
            len(campaigns)
        )

        campaign_open_count = (
            int(campaigns["opened"].sum())
            if not campaigns.empty
            else 0
        )

        campaign_click_count = (
            int(campaigns["clicked"].sum())
            if not campaigns.empty
            else 0
        )

        campaign_ctr = (
            campaign_click_count
            / campaign_sent_count
            if campaign_sent_count
            else 0.0
        )

        purchase_value = (
            int(purchase_events["value"].sum())
            if not purchase_events.empty
            else 0
        )

        learned_signal_score = int(
            sum(
                self.EVENT_WEIGHTS.get(
                    event_name,
                    0,
                )
                for event_name
                in events["event_name"]
            )
        )

        iphone18_score = int(
            sum(
                self.EVENT_WEIGHTS.get(
                    event_name,
                    0,
                )
                for event_name
                in product_events["event_name"]
            )
        )

        evidence_flags = {
            "saw_ad":
                bool(not ad_events.empty),

            "clicked_ad":
                bool(
                    not events[
                        events["event_name"].eq(
                            "ad_click"
                        )
                    ].empty
                ),

            "visited_mall":
                bool(not mall_events.empty),

            "tried_product":
                bool(
                    not events[
                        events["event_name"].eq(
                            "product_demo"
                        )
                    ].empty
                ),

            "compared_products":
                bool(
                    not comparison_events.empty
                ),

            "checked_price":
                bool(
                    not events[
                        events["event_name"].eq(
                            "price_check"
                        )
                    ].empty
                ),

            "added_to_cart":
                bool(
                    not events[
                        events["event_name"].eq(
                            "add_to_cart"
                        )
                    ].empty
                ),

            "started_checkout":
                bool(
                    not events[
                        events["event_name"].eq(
                            "checkout"
                        )
                    ].empty
                ),

            "abandoned_cart":
                bool(
                    not events[
                        events["event_name"].eq(
                            "cart_abandon"
                        )
                    ].empty
                ),

            "purchased":
                bool(
                    not purchase_events.empty
                ),

            "contacted_support":
                bool(
                    not support_events.empty
                ),

            "shared_or_referred":
                bool(
                    not events[
                        events["event_name"].isin(
                            [
                                "social_share",
                                "referral",
                            ]
                        )
                    ].empty
                ),
        }

        return {
            "customer_status":
                experience.profile["status"],

            "profile_type":
                experience.profile["profile_type"],

            "ltv_vnd":
                int(
                    experience.profile["ltv"] or 0
                ),

            "current_product":
                self.config.CURRENT_PRODUCT,

            "event_count":
                int(len(events)),

            "recent_30d_event_count":
                int(len(recent_30d)),

            "purchase_count":
                int(len(purchase_events)),

            "purchase_value_vnd":
                purchase_value,

            "iphone18_interactions":
                int(len(product_events)),

            "current_product_interactions":
                int(len(current_product_events)),

            "comparison_count":
                int(len(comparison_events)),

            "ad_interaction_count":
                int(len(ad_events)),

            "mall_interaction_count":
                int(len(mall_events)),

            "high_intent_event_count":
                int(len(high_intent_events)),

            "support_ticket_count":
                int(len(tickets)),

            "support_event_count":
                int(len(support_events)),

            "campaign_sends":
                campaign_sent_count,

            "campaign_opens":
                campaign_open_count,

            "campaign_clicks":
                campaign_click_count,

            "campaign_ctr":
                round(
                    campaign_ctr,
                    3,
                ),

            "last_event":
                (
                    last_event.strftime(
                        "%Y-%m-%d %H:%M"
                    )
                    if last_event is not None
                    else None
                ),

            "days_since_last_event":
                days_since_last_event,

            "learned_signal_score":
                learned_signal_score,

            "iphone18_interest_score":
                iphone18_score,

            "funnel_stage":
                self._infer_funnel_stage(
                    evidence_flags
                ),

            "behavior_cluster":
                self._infer_behavior_cluster(
                    evidence_flags,
                    experience.profile[
                        "status"
                    ],
                    experience.segments,
                ),

            "evidence_flags":
                evidence_flags,
        }

    @staticmethod
    def _events_by_name(
        events: pd.DataFrame,
        names: List[str],
    ) -> pd.DataFrame:

        if events.empty:
            return events

        return events[
            events["event_name"].isin(
                names
            )
        ]

    @staticmethod
    def _infer_funnel_stage(
        flags: Dict[str, bool],
    ) -> str:

        if flags["purchased"]:
            return "Customer / Converted"

        if flags["abandoned_cart"]:
            return "Checkout / Abandoned"

        if flags["started_checkout"]:
            return "Checkout"

        if flags["added_to_cart"]:
            return "Cart"

        if flags["tried_product"]:
            return "Consideration / Demo"

        if flags["compared_products"]:
            return "Consideration / Comparison"

        if flags["clicked_ad"]:
            return "Interest / Click"

        if flags["saw_ad"]:
            return "Awareness"

        return "Unknown"

    @staticmethod
    def _infer_behavior_cluster(
        flags: Dict[str, bool],
        status: str,
        segments: List[str],
    ) -> str:

        segment_text = (
            " ".join(segments).lower()
        )

        if (
            flags["abandoned_cart"]
            and flags["started_checkout"]
        ):
            return "High Intent / Friction"

        if (
            "price sensitive"
            in segment_text
        ):
            return "Price Sensitive"

        if (
            "churn risk"
            in segment_text
        ):
            return "Retention Risk"

        if (
            "cross-sell opportunity"
            in segment_text
        ):
            return "Cross-sell"

        if (
            "brand advocate"
            in segment_text
        ):
            return "Advocate"

        if flags["shared_or_referred"]:
            return "Advocate / Referral"

        if status == "customer":
            return "Existing Customer"

        return "Prospect"


# ============================================================
# 6. Rich context builder
# ============================================================

class RichContextBuilder:
    """
    Combines raw customer evidence + deterministic analytics into
    one structured customer context before the LLM is called.
    """

    def __init__(self, config: DemoConfig):
        self.config = config

    def build(
        self,
        experience: CustomerExperience,
        learned_model: Dict[str, Any],
    ) -> Dict[str, Any]:

        events = experience.events.copy()
        campaigns = experience.campaigns.copy()
        tickets = experience.tickets.copy()

        return {
            "analysis": {
                "as_of":
                    self.config.AS_OF,

                "product":
                    self.config.PRODUCT,

                "current_product":
                    self.config.CURRENT_PRODUCT,

                "objective":
                    (
                        "Understand the current customer context "
                        "and determine a measurable next-best action."
                    ),
            },

            "customer": {
                "id":
                    experience.profile[
                        "customer_id"
                    ],

                "name":
                    experience.profile[
                        "full_name"
                    ],

                "city":
                    experience.profile[
                        "city"
                    ],

                "status":
                    experience.profile[
                        "status"
                    ],

                "profile_type":
                    experience.profile[
                        "profile_type"
                    ],

                "ltv_vnd":
                    int(
                        experience.profile["ltv"]
                        or 0
                    ),

                "segments":
                    experience.segments,
            },

            "learned_model":
                learned_model,

            "event_summary":
                self._event_summary(
                    events
                ),

            "channel_summary":
                self._channel_summary(
                    events
                ),

            "product_summary":
                self._product_summary(
                    events
                ),

            "comparison_summary":
                self._comparison_summary(
                    events
                ),

            "campaign_summary":
                self._campaign_summary(
                    campaigns
                ),

            "support_summary":
                self._support_summary(
                    tickets
                ),

            "recent_evidence":
                self._recent_timeline(
                    events,
                    limit=12,
                ),
        }

    @staticmethod
    def _event_summary(
        events: pd.DataFrame,
    ) -> Dict[str, int]:

        if events.empty:
            return {}

        return (
            events["event_name"]
            .value_counts()
            .to_dict()
        )

    @staticmethod
    def _channel_summary(
        events: pd.DataFrame,
    ) -> Dict[str, Dict[str, int]]:

        if events.empty:
            return {}

        summary = (
            events.groupby(
                "channel"
            )
            .agg(
                events=(
                    "event_name",
                    "count",
                ),
                value=(
                    "value",
                    "sum",
                ),
            )
            .reset_index()
        )

        return {
            row["channel"]: {
                "events":
                    int(
                        row["events"]
                    ),
                "value":
                    int(
                        row["value"]
                    ),
            }
            for _, row
            in summary.iterrows()
        }

    def _product_summary(
        self,
        events: pd.DataFrame,
    ) -> Dict[str, Dict[str, int]]:

        if events.empty:
            return {}

        product_events = (
            events[
                events["product"].notna()
            ]
            .groupby(
                "product"
            )
            .agg(
                interactions=(
                    "event_name",
                    "count",
                ),
                purchases=(
                    "event_name",
                    lambda values:
                        int(
                            (
                                values
                                == "purchase"
                            ).sum()
                        ),
                ),
                value=(
                    "value",
                    "sum",
                ),
            )
            .reset_index()
        )

        return {
            row["product"]: {
                "interactions":
                    int(
                        row["interactions"]
                    ),
                "purchases":
                    int(
                        row["purchases"]
                    ),
                "value":
                    int(
                        row["value"]
                    ),
            }
            for _, row
            in product_events.iterrows()
        }

    @staticmethod
    def _comparison_summary(
        events: pd.DataFrame,
    ) -> Dict[str, Any]:

        if events.empty:
            return {
                "comparison_events": 0,
                "comparison_details": [],
            }

        comparisons = events[
            events["event_name"].isin(
                [
                    "product_compare",
                    "comparison_content",
                ]
            )
        ]

        return {
            "comparison_events":
                int(len(comparisons)),

            "comparison_details":
                comparisons[
                    [
                        "event_time",
                        "channel",
                        "product",
                        "details",
                    ]
                ]
                .astype(str)
                .to_dict(
                    orient="records"
                ),
        }

    @staticmethod
    def _campaign_summary(
        campaigns: pd.DataFrame,
    ) -> Dict[str, Any]:

        if campaigns.empty:
            return {
                "sends": 0,
                "opens": 0,
                "clicks": 0,
                "campaigns": [],
            }

        return {
            "sends":
                int(len(campaigns)),

            "opens":
                int(
                    campaigns["opened"].sum()
                ),

            "clicks":
                int(
                    campaigns["clicked"].sum()
                ),

            "campaigns":
                campaigns[
                    [
                        "campaign_name",
                        "channel",
                        "sent_at",
                        "opened",
                        "clicked",
                    ]
                ]
                .astype(str)
                .to_dict(
                    orient="records"
                ),
        }

    @staticmethod
    def _support_summary(
        tickets: pd.DataFrame,
    ) -> Dict[str, Any]:

        if tickets.empty:
            return {
                "ticket_count": 0,
                "recent_tickets": [],
            }

        recent = tickets.tail(5)

        return {
            "ticket_count":
                int(len(tickets)),

            "recent_tickets":
                recent[
                    [
                        "channel",
                        "subject",
                        "message",
                        "created_at",
                    ]
                ]
                .astype(str)
                .to_dict(
                    orient="records"
                ),
        }

    @staticmethod
    def _recent_timeline(
        events: pd.DataFrame,
        limit: int = 12,
    ) -> List[Dict[str, Any]]:

        if events.empty:
            return []

        recent = events.tail(limit)

        return (
            recent[
                [
                    "event_time",
                    "event_name",
                    "channel",
                    "product",
                    "value",
                    "details",
                ]
            ]
            .astype(str)
            .to_dict(
                orient="records"
            )
        )


# ============================================================
# 7. Persona Decision Engine
# ============================================================

class PersonaDecisionEngine:
    """
    LLM reasoning layer.

    Rich context is generated BEFORE the LLM is called.
    The LLM does not query SQLite and does not receive raw tables.
    """

    JSON_SCHEMA = """
{
  "kernel_lens": {
    "persona_name": "...",
    "lens": "...",
    "signals": ["...", "...", "..."],
    "confidence": "High|Medium|Low"
  },
  "perception": {
    "current_state": "...",
    "what_matters_now": ["...", "..."],
    "confidence": "High|Medium|Low"
  },
  "interpretation": {
    "meaning_for_business": "...",
    "risk_or_opportunity": "...",
    "reasoning": "..."
  },
  "action": {
    "next_best_action": "...",
    "channel": "...",
    "why": "...",
    "success_signal": "..."
  }
}
""".strip()

    SYSTEM_PROMPT = """
You are LEO, an AI Customer Context and Decision Intelligence platform.

Your job is to transform structured Customer 360 evidence into a
grounded Persona Decision Loop:

Experience
→ Learned Model
→ Kernel / Lens
→ Perception
→ Interpretation
→ Action
→ New Experience

The input has already been processed from SQLite into a rich context.
Treat the supplied context as the only source of truth.

IMPORTANT PRINCIPLES

1. Observed behavior is evidence.
2. Learned Model contains deterministic metrics computed from data.
3. Persona / Kernel / Lens is an inference from evidence, not a fact.
4. Perception describes the customer's current state.
5. Interpretation explains the business meaning of that state.
6. Action must be measurable through future customer events.
7. Do not invent product specifications.
8. Do not infer age, gender, income, motivation, or preferences
   unless directly supported by evidence.
9. Seeing an advertisement means awareness, not purchase intent.
10. Clicking an advertisement indicates interest, not conversion.
11. A shopping mall visit indicates physical experience, not conversion.
12. A product demo indicates consideration, not purchase intent by itself.
13. Comparison behavior is evidence that the customer is evaluating
    alternatives.
14. A checkout event indicates stronger intent than an ad click.
15. Cart abandonment indicates intent plus friction, not rejection.
16. Existing customer status does not mean the customer wants to upgrade.
17. A purchase of another product does not equal an iPhone 18 purchase.
18. Never use future outcomes unless they are explicitly supplied.
19. Keep the answer concise and useful for Marketing, Sales, CRM,
    and Customer Success teams.

For this demo, the current phone is iPhone 15.

The key question is NOT:
"How do we sell iPhone 18?"

The key question is:
"What does the customer evidence tell us about whether an iPhone 18
upgrade is meaningful for THIS customer right now?"

If evidence suggests insufficient incremental value versus iPhone 15,
it is valid to conclude:

"The customer does not currently perceive iPhone 18 as sufficiently
different from iPhone 15 to justify an upgrade."

A valid action in that situation is:

"Keep using iPhone 15; do not push an upgrade until a stronger
upgrade signal appears."

Do not force an upgrade recommendation when evidence does not support it.

Return ONLY one valid JSON object matching the required schema.
""".strip()

    def __init__(
        self,
        config: DemoConfig,
    ):
        self.config = config

        self.llm = get_llm(
            config.LLM_ALIAS,
            temperature=config.LLM_TEMPERATURE,
            max_tokens=config.LLM_MAX_TOKENS,
        )

        self.prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    self.SYSTEM_PROMPT,
                ),
                (
                    "human",
                    """
Analyze this customer context.

RICH CUSTOMER CONTEXT:
{context}

REQUIRED JSON SCHEMA:
{json_schema}

Return JSON only.
""",
                ),
            ]
        )

    def analyze(
        self,
        rich_context: Dict[str, Any],
    ) -> Dict[str, Any]:

        context_json = json.dumps(
            rich_context,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

        response = (
            self.prompt
            | self.llm
            | StrOutputParser()
        ).invoke(
            {
                "context":
                    context_json,

                "json_schema":
                    self.JSON_SCHEMA,
            }
        )

        result = parse_json_safely(
            response
        )

        if result.get("_parse_error"):
            return self._fallback(
                rich_context
            )

        return result

    def _fallback(
        self,
        rich_context: Dict[str, Any],
    ) -> Dict[str, Any]:

        learned = (
            rich_context[
                "learned_model"
            ]
        )

        flags = (
            learned[
                "evidence_flags"
            ]
        )

        compared = (
            flags["compared_products"]
        )

        high_intent = (
            flags["added_to_cart"]
            or flags["started_checkout"]
        )

        purchased = (
            flags["purchased"]
            and learned[
                "iphone18_interactions"
            ] > 0
        )

        if purchased:

            persona_name = (
                "Converted iPhone 18 Customer"
            )

            lens = (
                "The customer has moved from evaluation "
                "to confirmed purchase behavior."
            )

            perception = (
                "The customer has demonstrated sufficient "
                "purchase intent to convert."
            )

            interpretation = (
                "The iPhone 18 proposition is currently "
                "relevant to this customer."
            )

            action = (
                "Confirm post-purchase onboarding "
                "and retention journey."
            )

        elif high_intent:

            persona_name = (
                "High Intent with Purchase Friction"
            )

            lens = (
                "The customer appears focused on completing "
                "a purchase but has not yet converted."
            )

            perception = (
                "The customer is close to conversion but "
                "friction is visible in the journey."
            )

            interpretation = (
                "The opportunity is to understand and remove "
                "the specific friction rather than increase awareness."
            )

            action = (
                "Trigger a friction-resolution journey "
                "and measure checkout recovery."
            )

        elif compared:

            persona_name = (
                "Value Before Upgrade"
            )

            lens = (
                "The customer evaluates iPhone 18 through "
                "the lens of incremental value versus iPhone 15."
            )

            perception = (
                "The customer is actively evaluating the "
                "difference between the current device and "
                "the new product."
            )

            interpretation = (
                "The current evidence does not prove that the "
                "customer sees enough incremental value to justify upgrading."
            )

            action = (
                "Keep using iPhone 15; do not push an upgrade "
                "until a stronger upgrade signal appears."
            )

        else:

            persona_name = (
                "Early Interest"
            )

            lens = (
                "The customer has shown awareness or early "
                "interest but has not provided strong upgrade evidence."
            )

            perception = (
                "The customer is aware of iPhone 18 but current "
                "engagement is still early."
            )

            interpretation = (
                "The customer is not yet sufficiently qualified "
                "for aggressive upgrade messaging."
            )

            action = (
                "Continue low-pressure education and observe "
                "for stronger behavioral signals."
            )

        return {
            "kernel_lens": {
                "persona_name":
                    persona_name,

                "lens":
                    lens,

                "signals": [
                    (
                        "iPhone 18 interactions: "
                        f"{learned['iphone18_interactions']}"
                    ),
                    (
                        "Comparison events: "
                        f"{learned['comparison_count']}"
                    ),
                    (
                        "Funnel stage: "
                        f"{learned['funnel_stage']}"
                    ),
                ],

                "confidence":
                    "Medium",
            },

            "perception": {
                "current_state":
                    perception,

                "what_matters_now": [
                    "Meaningful customer value",
                    "Observable next-step intent",
                ],

                "confidence":
                    "Medium",
            },

            "interpretation": {
                "meaning_for_business":
                    interpretation,

                "risk_or_opportunity":
                    (
                        "Avoid over-targeting when customer evidence "
                        "does not support the action."
                    ),

                "reasoning":
                    (
                        "Fallback interpretation generated from "
                        "deterministic customer evidence."
                    ),
            },

            "action": {
                "next_best_action":
                    action,

                "channel":
                    "CRM / Personalized Experience",

                "why":
                    (
                        "Align the next interaction with the "
                        "customer's observed state."
                    ),

                "success_signal":
                    (
                        "A measurable future behavior confirms "
                        "whether the action was relevant."
                    ),
            },
        }


# ============================================================
# 8. Colab renderer
# ============================================================

class ColabRenderer:

    HEADER_STYLE = [
        {
            "selector": "th",
            "props": [
                (
                    "background-color",
                    "#172554",
                ),
                (
                    "color",
                    "white",
                ),
                (
                    "font-weight",
                    "bold",
                ),
                (
                    "text-align",
                    "left",
                ),
                (
                    "padding",
                    "8px",
                ),
            ],
        }
    ]

    BODY_STYLE = {
        "font-size": "12px",
        "white-space": "normal",
        "text-align": "left",
        "padding": "7px",
        "vertical-align": "top",
    }

    @classmethod
    def style_table(
        cls,
        df: pd.DataFrame,
        max_width: str = "260px",
    ):

        return (
            df.style
            .hide(axis="index")
            .set_properties(
                **cls.BODY_STYLE
            )
            .set_table_styles(
                cls.HEADER_STYLE
                + [
                    {
                        "selector": "td",
                        "props": [
                            (
                                "max-width",
                                max_width,
                            )
                        ],
                    }
                ]
            )
        )

    @classmethod
    def render_database_stats(
        cls,
        database: DemoDatabase,
    ):

        display(
            Markdown(
                "## SQLite Demo Dataset"
            )
        )

        display(
            cls.style_table(
                database.table_counts(),
                max_width="220px",
            )
        )

    @classmethod
    def render_profile_summary(
        cls,
        results: List[Dict[str, Any]],
    ):

        rows = []

        for result in results:

            profile = result["profile"]
            learned = result["learned_model"]
            ai = result["ai_layers"]

            rows.append(
                {
                    "ID":
                        profile["customer_id"],

                    "Customer":
                        profile["full_name"],

                    "Scenario":
                        profile["profile_type"],

                    "Status":
                        profile["status"],

                    "Events":
                        learned["event_count"],

                    "iPhone 18 Interactions":
                        learned[
                            "iphone18_interactions"
                        ],

                    "Compare":
                        learned[
                            "comparison_count"
                        ],

                    "Funnel":
                        learned[
                            "funnel_stage"
                        ],

                    "Persona":
                        ai[
                            "kernel_lens"
                        ].get(
                            "persona_name",
                            "",
                        ),
                }
            )

        df = pd.DataFrame(
            rows
        )

        display(
            Markdown(
                "## Customer Profile Summary"
            )
        )

        display(
            cls.style_table(
                df,
                max_width="190px",
            )
        )

    @classmethod
    def render_decision_summary(
        cls,
        results: List[Dict[str, Any]],
    ):

        rows = []

        for result in results:

            profile = result["profile"]
            ai = result["ai_layers"]

            rows.append(
                {
                    "Customer":
                        profile["full_name"],

                    "Status":
                        profile["status"],

                    "Persona / Lens":
                        ai[
                            "kernel_lens"
                        ].get(
                            "persona_name",
                            "",
                        ),

                    "Perception":
                        ai[
                            "perception"
                        ].get(
                            "current_state",
                            "",
                        ),

                    "Interpretation":
                        ai[
                            "interpretation"
                        ].get(
                            "meaning_for_business",
                            "",
                        ),

                    "Next Best Action":
                        ai[
                            "action"
                        ].get(
                            "next_best_action",
                            "",
                        ),

                    "Confidence":
                        ai[
                            "kernel_lens"
                        ].get(
                            "confidence",
                            "",
                        ),
                }
            )

        df = pd.DataFrame(
            rows
        )

        display(
            Markdown(
                "## Persona → Perception → Interpretation → Action"
            )
        )

        display(
            cls.style_table(
                df,
                max_width="300px",
            )
        )

    @classmethod
    def render_evidence(
        cls,
        result: Dict[str, Any],
    ):

        profile = result["profile"]
        learned = result["learned_model"]
        context = result["rich_context"]

        display(
            Markdown(
                f"""
### {profile["full_name"]}

**Scenario:** `{profile["profile_type"]}`
**Status:** `{profile["status"]}`
**Current device:** `{DemoConfig.CURRENT_PRODUCT}`
**Funnel stage:** `{learned["funnel_stage"]}`
**Behavior cluster:** `{learned["behavior_cluster"]}`
"""
            )
        )

        evidence_df = pd.DataFrame(
            [
                {
                    "Metric":
                        "Total events",
                    "Value":
                        learned["event_count"],
                },
                {
                    "Metric":
                        "Recent 30d events",
                    "Value":
                        learned[
                            "recent_30d_event_count"
                        ],
                },
                {
                    "Metric":
                        "iPhone 18 interactions",
                    "Value":
                        learned[
                            "iphone18_interactions"
                        ],
                },
                {
                    "Metric":
                        "Comparison events",
                    "Value":
                        learned[
                            "comparison_count"
                        ],
                },
                {
                    "Metric":
                        "Ad interactions",
                    "Value":
                        learned[
                            "ad_interaction_count"
                        ],
                },
                {
                    "Metric":
                        "Mall interactions",
                    "Value":
                        learned[
                            "mall_interaction_count"
                        ],
                },
                {
                    "Metric":
                        "High-intent events",
                    "Value":
                        learned[
                            "high_intent_event_count"
                        ],
                },
                {
                    "Metric":
                        "Campaign CTR",
                    "Value":
                        f"{learned['campaign_ctr']:.1%}",
                },
            ]
        )

        display(
            cls.style_table(
                evidence_df,
                max_width="220px",
            )
        )

        display(
            Markdown(
                "#### Recent evidence"
            )
        )

        recent_df = pd.DataFrame(
            context["recent_evidence"]
        )

        if not recent_df.empty:

            display(
                cls.style_table(
                    recent_df,
                    max_width="260px",
                )
            )

    @classmethod
    def render_future_outcomes(
        cls,
        results: List[Dict[str, Any]],
    ):

        rows = []

        for result in results:

            outcome = result.get(
                "future_outcome"
            )

            if not outcome:
                continue

            rows.append(
                {
                    "Customer":
                        result["profile"][
                            "full_name"
                        ],

                    "Initial Status":
                        result["profile"][
                            "status"
                        ],

                    "Initial Decision":
                        result[
                            "ai_layers"
                        ][
                            "action"
                        ].get(
                            "next_best_action",
                            "",
                        ),

                    "Outcome Date":
                        outcome[
                            "outcome_date"
                        ],

                    "Outcome":
                        (
                            f"Purchased "
                            f"{outcome['product']}"
                        ),

                    "Final Status":
                        outcome[
                            "status_after"
                        ],

                    "Value":
                        (
                            f"{int(outcome['value']):,} "
                            "VND"
                        ),
                }
            )

        df = pd.DataFrame(
            rows
        )

        display(
            Markdown(
                "## Future Outcome — Profile 3"
            )
        )

        if df.empty:

            display(
                Markdown(
                    "No future outcomes are seeded."
                )
            )

            return

        display(
            cls.style_table(
                df,
                max_width="300px",
            )
        )


# ============================================================
# 9. Main application service
# ============================================================

class PersonaDecisionLoopDemo:
    """
    Main orchestration:

        SQLite
          ↓
        Pandas
          ↓
        Feature Engineering
          ↓
        Rich Context
          ↓
        LLM
          ↓
        Persona Decision Loop
    """

    def __init__(
        self,
        config: DemoConfig,
    ):

        self.config = config

        self.database = DemoDatabase(
            config.DB_PATH
        )

        self.scenario_factory = DemoScenarioFactory(
            config
        )

        self.feature_engineer = CustomerFeatureEngineer(
            config
        )

        self.context_builder = RichContextBuilder(
            config
        )

        self.decision_engine = PersonaDecisionEngine(
            config
        )

        self.repository: Optional[
            CustomerRepository
        ] = None

    def setup(self):

        customers = (
            self.scenario_factory.build()
        )

        self.database.connect()
        self.database.reset()
        self.database.create_schema()
        self.database.seed(
            customers
        )

        self.repository = CustomerRepository(
            self.database.connection,
            self.config,
        )

        return customers

    def run_customer(
        self,
        customer_id: int,
    ) -> Dict[str, Any]:

        if self.repository is None:
            raise RuntimeError(
                "Call demo.setup() before run_customer()."
            )

        # ----------------------------------------------------
        # Step 1 — Load raw evidence from SQLite into Pandas
        # ----------------------------------------------------

        experience = (
            self.repository.load_customer(
                customer_id=customer_id,
                as_of=self.config.AS_OF,
            )
        )

        # ----------------------------------------------------
        # Step 2 — Compute deterministic learned model
        # ----------------------------------------------------

        learned_model = (
            self.feature_engineer.build(
                experience
            )
        )

        # ----------------------------------------------------
        # Step 3 — Build rich context BEFORE LLM
        # ----------------------------------------------------

        rich_context = (
            self.context_builder.build(
                experience=experience,
                learned_model=learned_model,
            )
        )

        # ----------------------------------------------------
        # Step 4 — LLM reasons over rich context
        # ----------------------------------------------------

        ai_layers = (
            self.decision_engine.analyze(
                rich_context
            )
        )

        # ----------------------------------------------------
        # Step 5 — Future outcome is loaded separately
        # ----------------------------------------------------

        future_outcome = (
            self.repository.load_future_outcome(
                customer_id
            )
        )

        return {
            "profile":
                experience.profile,

            "learned_model":
                learned_model,

            "rich_context":
                rich_context,

            "ai_layers":
                ai_layers,

            "future_outcome":
                future_outcome,
        }

    def run_all(self):

        customers = (
            self.scenario_factory.build()
        )

        return [
            self.run_customer(
                customer.customer_id
            )
            for customer in customers
        ]


# ============================================================
# 10. Execute Task 9
# ============================================================

demo = PersonaDecisionLoopDemo(
    DemoConfig()
)

seeded_customers = demo.setup()

results = demo.run_all()


# ============================================================
# 11. Dataset overview
# ============================================================

display(
    Markdown(
        f"""
# TASK 9 — LEO Persona Decision Loop

**Product:** `{DemoConfig.PRODUCT}`
**Current device:** `{DemoConfig.CURRENT_PRODUCT}`
**Analysis date:** `{DemoConfig.AS_OF}`
**Profiles:** `{len(seeded_customers)}`

The SQLite dataset contains synthetic marketing scenarios including:

- Advertising exposure
- Ad clicks
- Shopping mall visits
- Product demos
- Product comparisons
- Price research
- Promotions
- Cart abandonment
- Customer support
- Churn risk
- Cross-sell
- Ad fatigue
- Competitor comparison
- Upgrade intent
- Seasonal / gift intent
- Referral / advocacy
- Conversion

The LLM receives the **computed rich context**, not the raw SQLite tables.
"""
    )
)

ColabRenderer.render_database_stats(
    demo.database
)


# ============================================================
# 12. Customer profile summary
# ============================================================

ColabRenderer.render_profile_summary(
    results
)


# ============================================================
# 13. Persona decision summary
# ============================================================

ColabRenderer.render_decision_summary(
    results
)


# ============================================================
# 14. Future outcome
# ============================================================

ColabRenderer.render_future_outcomes(
    results
)


# ============================================================
# 15. Detailed evidence + decision
# ============================================================

display(
    Markdown(
        """
## Detailed Persona Decision Loop

Experience
→ Learned Model
→ Kernel / Lens
→ Perception
→ Interpretation
→ Action
→ New Experience
"""
    )
)

for result in results:

    ColabRenderer.render_evidence(
        result
    )

    ai = result["ai_layers"]

    display(
        Markdown(
            f"""
#### Persona / Kernel / Lens

**Persona:**
{ai["kernel_lens"].get("persona_name", "")}

**Lens:**
{ai["kernel_lens"].get("lens", "")}

**Signals:**
{", ".join(ai["kernel_lens"].get("signals", []))}

**Confidence:**
{ai["kernel_lens"].get("confidence", "")}

#### Perception

{ai["perception"].get("current_state", "")}

**What matters now:**
{", ".join(ai["perception"].get("what_matters_now", []))}

**Confidence:**
{ai["perception"].get("confidence", "")}

#### Interpretation

**Business meaning:**
{ai["interpretation"].get("meaning_for_business", "")}

**Risk / Opportunity:**
{ai["interpretation"].get("risk_or_opportunity", "")}

**Reasoning:**
{ai["interpretation"].get("reasoning", "")}

#### Action

**Next Best Action:**
{ai["action"].get("next_best_action", "")}

**Channel:**
{ai["action"].get("channel", "")}

**Why:**
{ai["action"].get("why", "")}

**Success Signal:**
{ai["action"].get("success_signal", "")}
"""
        )
    )

    display(
        Markdown("---")
    )


# ============================================================
# 16. Rich context example — Profile 3
# ============================================================

profile_3 = next(
    result
    for result in results
    if result["profile"]["customer_id"] == 1003
)

display(
    Markdown(
        """
## Example — Rich Context Sent to the LLM

The important architecture is:

Raw Experience
→ deterministic feature computation
→ rich customer context
→ LLM reasoning

The six-month future outcome is intentionally NOT included in this
initial context.
"""
    )
)

print(
    json.dumps(
        profile_3["rich_context"],
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)


# ============================================================
# 17. Profile 3 — six-month outcome
# ============================================================

display(
    Markdown(
        """
## Profile 3 — Six-Month Learning Loop

Initial state:

Lead
→ iPhone 18 awareness
→ Ad engagement
→ Shopping mall experience
→ Product comparison
→ No immediate upgrade signal

Six months later:

New Experience
→ iPhone 18 purchase
→ Customer

The future outcome is stored in SQLite separately and can later become
new evidence for the next Persona / Learned Model cycle.
"""
    )
)

print(
    json.dumps(
        profile_3["future_outcome"],
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)



## Summary

| #     | Use case                                                        | Model(s)                                                    | Description                                                                                                                                                                                                                                                                                                                                                                                                                 |
| ----- | --------------------------------------------------------------- | ----------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1     | Customer Support                                                | `deepseek/deepseek-chat`                                    | RAG over a help-center knowledge base, with a Tavily domain-restricted web search fallback when the answer cannot be found in the KB                                                                                                                                                                                                                                                                                        |
| 2     | Text-to-SQL                                                     | `deepseek/deepseek-v4-pro`                                  | Schema-grounded RAG → SQL generation → SQL execution against a small sample CDP database                                                                                                                                                                                                                                                                                                                                    |
| 3     | Marketing Automation — Campaign Orchestration                   | `moonshotai/kimi-k3`                                        | A RAG-grounded, tool-calling agent that plans and executes a multi-step campaign workflow                                                                                                                                                                                                                                                                                                                                   |
| 4     | Marketing Automation — Copy Drafting                            | `~deepseek/deepseek-v4-flash-latest`                              | Brand-guideline-grounded structured copy generation, including subject line, preview text, email body, and social caption                                                                                                                                                                                                                                                                                                   |
| 5     | Ask Your Data — Analytics Chat                                  | `deepseek/deepseek-v4-pro` + `~deepseek/deepseek-v4-flash-latest` | A conversational layer on top of the Text-to-SQL workflow: multi-turn analysis, maintained query context (e.g. "Now break that down by city"), chart-ready results                                                                                                                                                                                                                                                          |
| 6     | Ticket Classification                                           | `nvidia/nemotron-3-nano-30b-a3b`                            | High-volume classification of support tickets by urgency, sentiment, and routing team — a good fit for a smaller, lower-cost model since it's classification, not open-ended generation                                                                                                                                                                                                                                     |
| 7     | Campaign Performance Narrative                                  | `~deepseek/deepseek-v4-flash-latest`                              | Converts the numeric output of a Text-to-SQL query into a concise, plain-English performance summary for non-technical stakeholders                                                                                                                                                                                                                                                                                         |
| 8     | Schema-Mapping Assistant — Client Onboarding                    | `z-ai/glm-5.3`                                              | RAG over a client's raw source columns and CDP schema documentation to propose field mappings, flagging ambiguous ones for human review instead of silently guessing                                                                                                                                                                                                                                                        |
| 9 | Persona Decision Loop — Customer Context & Next Best Action | `~deepseek/deepseek-v4-flash-latest`                              | Builds a grounded dynamic Persona from Customer 360 evidence, then derives Perception → Interpretation → Action. Combines behavioral events, campaign interactions, shopping-mall/offline experience, product comparisons, and customer history to determine the current customer context and recommend a measurable next-best action. The Persona is continuously updated when new customer outcomes become available. |

### How the use cases build on each other
- Tasks 2, 5, 7, and 8 all share the **same `schema_retriever`** built once in Task 2 — a new use case over the same data doesn't need a new schema index.
- Tasks 5 and 7 share the **same composition pattern**: a strong model generates/executes SQL, then a cheap model turns the table into prose.
- Task 7 calls Task 2's `run_text_to_sql` directly rather than re-implementing SQL generation.

### Notes for production use
- All sample data (customers, segments, campaigns, events, tickets, playbooks, brand guidelines) is synthetic — swap in your real KB pages, real database connection, and real policy/brand docs.
- Swap any model in the `MODELS` dict at the top without touching downstream code.
- The Text-to-SQL guard (Tasks 2, 5, 7) only blocks a few obvious write keywords — for production, run generated SQL through a read-only DB user/role, not just a keyword filter.
- Task 3's orchestration tools are mocked — wire `send_test_message` and `schedule_campaign` to your real ESP/CDP APIs when ready.
- Consider prompt-caching your system prompts/KB context if you move this to production — it meaningfully cuts cost on the RAG- and playbook-heavy tasks (1, 3, 6), which repeat the same context on every call.